# MV and Cluster Analysis Template
*For Cai Lab **NanoMAP data analysis workflow**. Created by: En Cai. This version was edited on 2026/07/31, and tested with **Python 3.13.5.***

**This script loads Imaris exported statistics of objects (cell, receptor clusters, microvilli, etc.) from a given dataset, and calculates and plots properties and coloc/overlap of receptors and microvilli.**

The template script uses **'markerA'** and **'markerB'** for receptors, which need to be **replaced** with the names of receptors and corresponding channels to generate a **dataset-specitic script** for running.  

**How to use:**
* Step 1 - Run the **MV_and_Cluster_parameter.ipynb** to modify this templet script to match the specific dataset you want to analyze. 
* Step 2 - In the newly generated script, set the folder_path containing the data, and names of .xls files in the modified script and run the script. 

## 1. User Input:

In [ ]:
# Set path (will be overridden by papermill)
path = "./" # <-- replace with the path to your data folder 

# Load data from the .xls files

# Receptors
markerA_All_data_file = 'markerA.xls'   # <-- replace with the name of your markerA data file (All surfaces)
markerB_All_data_file = 'markerB.xls'   # <-- replace with the name of your markerB data file (All surfaces)

# Microvilli
MV_data_file = 'MV.xls'      # <-- replace with the name of your MV data file

# Receptors in microvilli
markerA_select_data_file = 'markerA_MV.xls'  # <-- replace with the name of your markerA in microvilli data file
markerB_select_data_file = 'markerB_MV.xls' # <-- replace with the name of your markerB in microvilli data file

# Cell object
cell_area_file = 'Cell.xls'  # <-- replace with the name of your Cell_info data file
cell_SA_file = 'Cell_Surface.xls'  # <-- replace with the name of your Cell_surface data file

# Save results in the original folder (no new folder created)
result_folder_path = path

## 2. Import packages

In [ ]:
import numpy as np
import pandas as pd
import os
import xlrd

import matplotlib.pyplot as plt
import scipy.stats as stats
import seaborn as sns
from statannotations.Annotator import Annotator

sns.set_context('notebook')
sns.set_style('darkgrid')

## 3. Load Data 
* Function: load xls as df
  * load_xls_as_df(file_path, extract_map)

In [ ]:
def load_xls_as_df(file_path, extract_map):
    """
    Extract and rename multiple columns from multiple sheets in an .xls file.

    Parameters:
    - file_path (str): Path to the .xls file.
    - extract_map (list of dict): Each dict should have:
        {
            'sheet': sheet index or name,
            'columns': {
                'original_column_name1': 'new_column_name1',
                'original_column_name2': 'new_column_name2',
                ...
            }
        }

    Returns:
    - pd.DataFrame: Combined DataFrame with all specified and renamed columns.
    """
    all_parts = []

    for item in extract_map:
        sheet = item['sheet']
        column_map = item['columns']
        try:
            # Read only the needed columns
            df = pd.read_excel(file_path, sheet_name=sheet, skiprows=1, usecols=list(column_map.keys()), engine='xlrd')
            # Rename columns
            df = df.rename(columns=column_map)
            all_parts.append(df)
        except Exception as e:
            print(f"Error reading from sheet {sheet}: {e}")
            
    # Concatenate all selected columns horizontally (side-by-side)
    combined_df = pd.concat(all_parts, axis=1)
    return combined_df

## 4. Cell Statistics

### 4.1 Cell area

In [ ]:
# Load the workbook
Cell_area = os.path.join(path, cell_area_file)
Cell_area_xls = xlrd.open_workbook(Cell_area)

# get the Cell area sheet
sheet = Cell_area_xls.sheet_by_name('Cell Area')

# Get value at cell A3 (row 2, column 0) – note: xlrd uses 0-based indexing
cell_area = sheet.cell_value(rowx=2, colx=0)

# Inline output:
print("Cell area:", cell_area)

### 4.2 Cell surface area

In [ ]:
# Load the workbook
Cell_surf_area = os.path.join(path, cell_SA_file)
Cell_surf_area_xls = xlrd.open_workbook(Cell_surf_area)

# get the Cell area sheet
sheet = Cell_surf_area_xls.sheet_by_name('Area')

# Get value at cell A3 (row 2, column 0) – note: xlrd uses 0-based indexing
cell_surf_area = sheet.cell_value(rowx=2, colx=0)

# Inline output:
print("Cell surface area:", cell_surf_area)

## 5. markerA statistics for all detected clusters

### 5.1 Load sheet_data for All markerA surfaces
* Save as df_markerA_all_sheet_data
  * DataFrame for markerA with overlapped volume ratio, overlapped volume, and shortest distance to surfaces for both markerB and MV
* Check for sheet names in df_markerA_all_sheet_data

In [ ]:
# File Name
markerA_All = os.path.join(path, markerA_All_data_file)

# ------------Load data from the .xls file------------------------
markerA_all_xls = xlrd.open_workbook(markerA_All)

# List to store results
markerA_all_sheets = []

# Iterate over all sheets
for sheet_name in markerA_all_xls.sheet_names():
    sheet = markerA_all_xls.sheet_by_name(sheet_name)
    
    # Safely try to get the first cell
    try:
        first_cell = sheet.cell_value(0, 0)
    except IndexError:
        first_cell = None
    
    markerA_all_sheets.append({'Sheet Name': sheet_name, 'Data Name': first_cell})

# ------------Save loaded data in a DataFrame----------------------------------
# Create a DataFrame
df_markerA_all_sheets = pd.DataFrame(markerA_all_sheets)

# Find which sheet to read based on the data-content
markerA_olvr_markerB_sheet_name = df_markerA_all_sheets[
df_markerA_all_sheets['Data Name'].astype(str).str.contains('Overlapped Volume Ratio to Surfaces', na=False) &
df_markerA_all_sheets['Data Name'].astype(str).str.contains('markerB', na=False) & 
df_markerA_all_sheets['Data Name'].astype(str).str.contains('Selection', na=False)
]

# Find which sheet to read based on the data-content
markerA_olvr_MV_sheet_name = df_markerA_all_sheets[
df_markerA_all_sheets['Data Name'].astype(str).str.contains('Overlapped Volume Ratio to Surfaces', na=False) &
df_markerA_all_sheets['Data Name'].astype(str).str.contains('MV', na=False)
]

# Find which sheet to read based on the data-content
markerA_ol_V_markerB_sheet_name = df_markerA_all_sheets[
df_markerA_all_sheets['Data Name'].astype(str).str.contains('Overlapped Volume to Surfac', na=False) &
df_markerA_all_sheets['Data Name'].astype(str).str.contains('markerB', na=False) & 
df_markerA_all_sheets['Data Name'].astype(str).str.contains('Selection', na=False)
]

# Find which sheet to read based on the data-content
markerA_ol_V_MV_sheet_name = df_markerA_all_sheets[
df_markerA_all_sheets['Data Name'].astype(str).str.contains('Overlapped Volume to Surfaces', na=False) &
df_markerA_all_sheets['Data Name'].astype(str).str.contains('MV', na=False)
]

#
markerA_shd_markerB_sheet_name = df_markerA_all_sheets[
df_markerA_all_sheets['Data Name'].astype(str).str.contains('Shortest Distance to Surfaces', na=False) &
df_markerA_all_sheets['Data Name'].astype(str).str.contains('markerB', na=False) & 
df_markerA_all_sheets['Data Name'].astype(str).str.contains('Selection', na=False)
]

#
markerA_shd_MV_sheet_name = df_markerA_all_sheets[
df_markerA_all_sheets['Data Name'].astype(str).str.contains('Shortest Distance to Surfaces', na=False) &
df_markerA_all_sheets['Data Name'].astype(str).str.contains('MV', na=False)
]

# Create the DataFrame for markerA with overlapped volume ratio, overlapped volume, and shortest distance to surfaces for both markerB and MV
df_markerA_all_sheet_data = pd.concat([markerA_olvr_markerB_sheet_name, markerA_olvr_MV_sheet_name,
                                  markerA_ol_V_markerB_sheet_name, markerA_ol_V_MV_sheet_name,
                                  markerA_shd_markerB_sheet_name, markerA_shd_MV_sheet_name])

### 5.2 Load markerA stats by sheet in dataframe df_markerA_All

In [ ]:
# Enter Selected Sheets and Columns As Dictionaries
# Sheet 1 --> Surface ID
markerA_ID ={
    'sheet': 'Area', 
    'columns':{
        'ID': 'ID'
    }
}

# Sheet 2 --> Area
markerA_Area ={
    'sheet': 'Area', 
    'columns':{
        'Area': 'Area'
    }
}

# Sheet 3 --> markerA Intensity Mean (markerA channel --> Ch=mkrA )
markerA_Intensity_Mean = {
    'sheet': 'Intensity Mean Ch=mkrA Img=1',   # markerA channel --> Ch=1
    'columns': {
        'Intensity Mean': 'Intensity Mean markerA'
    }
}

# Sheet 4 --> markerA Intensity Mean (markerA channel --> Ch=mkrA )
markerA_Intensity_Sum = {
    'sheet': 'Intensity Sum Ch=mkrA Img=1',   # markerA channel --> Ch=mkrA 
    'columns': {
        'Intensity Sum': 'Intensity Sum markerA'
    }
}

# Sheet 5 --> Distance Transformation max from cell core (Distance Transformation channel --> Ch=DistTrans )
markerA_DT_max = {
    'sheet': 'Intensity Max Ch=DistTrans Img=1',  # channel --> Ch=DistTrans 
    'columns':{
        'Intensity Max': 'Distance to cell (max)'
    }
}

# Sheet 6 --> Distance Transformation min
markerA_DT_min = {
    'sheet': 'Intensity Min Ch=DistTrans Img=1',   # channel --> Ch=DistTrans 
    'columns':{
        'Intensity Min': 'Distance to cell (min)'
    }
}


# Sheet 7 --> Distance Transformation mean
markerA_DT_mean = {
    'sheet': 'Intensity Mean Ch=DistTrans Img=1',   # channel --> Ch=DistTrans 
    'columns':{
        'Intensity Mean': 'Distance to cell (mean)'
    }
}

# Sheet 8 --> Volume
markerA_V = {
    'sheet': 'Volume',
    'columns': {
        'Volume':'Volume'
    }
}

# Sheet 9 --> Voxel Size
markerA_Vox = {
    'sheet': 'Number of Voxels Img=1',
    'columns': {
        'Number of Voxels':'Number of Voxels'
    }
}

# Sheet 10 --> Intensity Sum in Channel binary MV (channel binary MV --> Ch=binary_MV )
markerA_MV = {
    'sheet': 'Intensity Sum Ch=binary_MV Img=1',
    'columns': {
        'Intensity Sum': 'Voxel Overlap with MV'
    }
}

# Sheets 11 --> Intensity Sum in Channel binary markerA (Ch=binary_mkrA )
markerA_markerA = {
    'sheet': 'Intensity Sum Ch=binary_mkrA Img=1',
    'columns': {
        'Intensity Sum': 'Voxel Overlap with markerA'
    }
}


# Sheets 12 --> Intensity Sum in Channel binary markerB (Ch=binary_mkrB )
markerA_markerB = {
    'sheet': 'Intensity Sum Ch=binary_mkrB Img=1',
    'columns': {
        'Intensity Sum': 'Voxel Overlap with markerB'
    }
}

# Sheet 13 --> Intensity Mean in Channel binary MV
markerA_MV_clust_vox = {
    'sheet': 'Intensity Mean Ch=binary_MV Img=1',
    'columns': {
        'Intensity Mean': 'Cluster Voxel Overlap with MV'
    }
}

# Sheet 14 --> Intensity Mean in Channel binary markerB (Ch=binary_mkrB)
markerA_markerB_clust_vox = {
    'sheet': 'Intensity Mean Ch=binary_mkrB Img=1',
    'columns': {
        'Intensity Mean': 'Cluster Voxel Overlap with markerB'
    }
}

# Sheet 15 --> get the intensity in markerB channel (Ch=mkrB )
markerA_int_mean_markerB = {
    'sheet': 'Intensity Mean Ch=mkrB Img=1',   # markerB channel --> Ch=mkrB 
    'columns': {
        'Intensity Mean': 'Intensity Mean (markerB Channel)'
    }
}

# Sheet 16 --> get the intensity sum in markerB channel (Ch=mkrB )
markerA_int_sum_markerB = {
    'sheet': 'Intensity Sum Ch=mkrB Img=1',   # markerB channel --> Ch=mkrB 
    'columns': {
        'Intensity Sum': 'Intensity Sum (markerB Channel)'
    }
}

##################### Surfaces ###################################################

# Sheet 17 --> get the overlapped volume ratio to surfaces=markerB_abv
markerA_olvr_markerB = {
    'sheet': markerA_olvr_markerB_sheet_name['Sheet Name'].tolist()[0],       # 'Overlapped Volume Ratio to -147',
    'columns':{
        'Overlapped Volume Ratio to Surfaces': 'Overlap Volume Ratio to markerB_MV'
    }
}

# Sheet 18 --> For markerA surfaces, get the overlapped volumne ratio to surfaces=MV
markerA_olvr_MV = {
    'sheet': markerA_olvr_MV_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Overlapped Volume Ratio to Surfaces': 'Overlap Volume Ratio to MV'
    }
}

# Sheet 19 --> For markerA surfaces, get the overlapped volumne to surfaces=markerB_abv
markerA_ol_V_markerB = {
    'sheet': markerA_ol_V_markerB_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Overlapped Volume to Surfaces': 'Overlap Volume to markerB_MV'
    }
}

# Sheet 20 --> For markerA surfaces, get the overlapped volumne to surfaces=MV_select
markerA_ol_V_MV = {
    'sheet': markerA_ol_V_MV_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Overlapped Volume to Surfaces': 'Overlap Volume to MV'
    }
}

# Sheet 21 --> get the Shortest Distance to surfaces=markerB_0_708
markerA_shd_markerB = {
    'sheet': markerA_shd_markerB_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Shortest Distance to Surfaces': 'Shortest Distance to markerB_MV'
    }
}

# Sheet 22 --> get the Shortest Distance to surfaces=MV_select
markerA_shd_MV = {
    'sheet': markerA_shd_MV_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Shortest Distance to Surfaces': 'Shortest Distance to MV'
    }
}

# ============Save sheet data in a DataFrame===========

# Create a list of all markerA data maps
markerA_data_map = [markerA_ID, markerA_Area, markerA_Intensity_Mean, markerA_Intensity_Sum, markerA_DT_max, markerA_DT_min,
                markerA_DT_mean, markerA_V, markerA_Vox, markerA_MV, markerA_markerB, markerA_markerA, markerA_MV_clust_vox, markerA_markerB_clust_vox,
                markerA_int_mean_markerB, markerA_int_sum_markerB, markerA_olvr_markerB, markerA_olvr_MV, markerA_ol_V_markerB, 
                markerA_ol_V_MV, markerA_shd_markerB, markerA_shd_MV]

# Add to DataFrame
df_markerA_All = load_xls_as_df(markerA_All, markerA_data_map)

# filter DataFrame to exclude voxels below 2
df_markerA_All = df_markerA_All[df_markerA_All['Number of Voxels'] > 2]

## 6. markerB statistics for all detected clusters

### 6.1 Load sheet_data for All markerB surfaces
* Save in dataframe df_markerB_all_sheet_data
* Check for sheet names in df_markerB_all_sheet_data

In [ ]:
# File Name
markerB_All = os.path.join(path, markerB_All_data_file)
# Load data from the .xls file
markerB_all_xls = xlrd.open_workbook(markerB_All)
# List to store results
markerB_all_sheets = []

# Iterate over all sheets
for sheet_name in markerB_all_xls.sheet_names():
    sheet = markerB_all_xls.sheet_by_name(sheet_name)
    
    # Safely try to get the first cell
    try:
        first_cell = sheet.cell_value(0, 0)
    except IndexError:
        first_cell = None
    
    markerB_all_sheets.append({'Sheet Name': sheet_name, 'Data Name': first_cell})

# Create a DataFrame
df_markerB_all_sheets = pd.DataFrame(markerB_all_sheets)

# Find which sheet to read based on the data-content
markerB_olvr_markerA_sheet_name = df_markerB_all_sheets[
df_markerB_all_sheets['Data Name'].astype(str).str.contains('Overlapped Volume Ratio to Surfaces', na=False) &
df_markerB_all_sheets['Data Name'].astype(str).str.contains('markerA', na=False) & 
df_markerB_all_sheets['Data Name'].astype(str).str.contains('Selection', na=False)
]

# Find which sheet to read based on the data-content
markerB_olvr_MV_sheet_name = df_markerB_all_sheets[
df_markerB_all_sheets['Data Name'].astype(str).str.contains('Overlapped Volume Ratio to Surfaces', na=False) &
df_markerB_all_sheets['Data Name'].astype(str).str.contains('MV', na=False)
]

# Find which sheet to read based on the data-content
markerB_ol_V_markerA_sheet_name = df_markerB_all_sheets[
df_markerB_all_sheets['Data Name'].astype(str).str.contains('Overlapped Volume to Surfac', na=False) &
df_markerB_all_sheets['Data Name'].astype(str).str.contains('markerA', na=False) & 
df_markerB_all_sheets['Data Name'].astype(str).str.contains('Selection', na=False)
]

# Find which sheet to read based on the data-content
markerB_ol_V_MV_sheet_name = df_markerB_all_sheets[
df_markerB_all_sheets['Data Name'].astype(str).str.contains('Overlapped Volume to Surfaces', na=False) &
df_markerB_all_sheets['Data Name'].astype(str).str.contains('MV', na=False)
]

markerB_shd_markerA_sheet_name = df_markerB_all_sheets[
df_markerB_all_sheets['Data Name'].astype(str).str.contains('Shortest Distance to Surfaces', na=False) &
df_markerB_all_sheets['Data Name'].astype(str).str.contains('markerA', na=False) & 
df_markerB_all_sheets['Data Name'].astype(str).str.contains('Selection', na=False)
]

markerB_shd_MV_sheet_name = df_markerB_all_sheets[
df_markerB_all_sheets['Data Name'].astype(str).str.contains('Shortest Distance to Surfaces', na=False) &
df_markerB_all_sheets['Data Name'].astype(str).str.contains('MV', na=False)
]

# Load data in a DataFrame containing overlapped volume ratio, overlapped volume, and shortest distance to surfaces for both markerA and MV
df_markerB_all_sheet_data = pd.concat([markerB_olvr_markerA_sheet_name, markerB_olvr_MV_sheet_name, 
                                     markerB_ol_V_markerA_sheet_name,markerB_ol_V_MV_sheet_name, 
                                    markerB_shd_markerA_sheet_name, markerB_shd_MV_sheet_name])

### 6.2 Load markerB stats by sheet in a dataframe
* save in dataframe: df_markerB_All

In [ ]:
# Enter Selected Sheets and Columns As Dictionaries

# Sheet 1 --> Surface ID
markerB_ID ={
    'sheet': 'Area', 
    'columns':{
        'ID': 'ID'
    }
}

# Sheet 2 --> Area
markerB_Area ={
    'sheet': 'Area', 
    'columns':{
        'Area': 'Area'
    }
}

# Sheet 3 --> markerB Intensity Mean (markerB channel --> Ch=mkrB )
markerB_Intensity_Mean = {
    'sheet': 'Intensity Mean Ch=mkrB Img=1',   # markerB channel --> Ch=mkrB 
    'columns': {
        'Intensity Mean': 'Intensity Mean markerB'
    }
}

# Sheet 4 --> markerB Intensity Mean (markerB channel --> Ch=mkrB )
markerB_Intensity_Sum = {
    'sheet': 'Intensity Sum Ch=mkrB Img=1',   # markerB channel --> Ch=mkrB 
    'columns': {
        'Intensity Sum': 'Intensity Sum markerB'
    }
}

# Sheet 5 --> Distance Transformation max from cell core (Distance Transformation channel --> Ch=DistTrans )
markerB_DT_max = {
    'sheet': 'Intensity Max Ch=DistTrans Img=1',  # channel --> Ch=DistTrans 
    'columns':{
        'Intensity Max': 'Distance to cell (max)'
    }
}

# Sheet 6 --> Distance Transformation min
markerB_DT_min = {
    'sheet': 'Intensity Min Ch=DistTrans Img=1',  # channel --> Ch=DistTrans 
    'columns':{
        'Intensity Min': 'Distance to cell (min)'
    }
}

# Sheet 7 --> Distance Transformation mean
markerB_DT_mean = {
    'sheet': 'Intensity Mean Ch=DistTrans Img=1',   # channel --> Ch=DistTrans 
    'columns':{
        'Intensity Mean': 'Distance to cell (mean)'
    }
}

# Sheet 8 --> Volume
markerB_V = {
    'sheet': 'Volume',
    'columns': {
        'Volume':'Volume'
    }
}

# Sheet 9 --> Voxel Size
markerB_Vox = {
    'sheet': 'Number of Voxels Img=1',
    'columns': {
        'Number of Voxels':'Number of Voxels'
    }
}

# Sheet 10 --> Intensity Sum in Channel binary MV (channel binary MV --> Ch=binary_MV )
markerB_MV = {
    'sheet': 'Intensity Sum Ch=binary_MV Img=1',
    'columns': {
        'Intensity Sum': 'Voxel Overlap with MV'
    }
}

# Sheets 11 --> Intensity Sum in Channel binary markerA (Ch=binary_mkrA )
markerB_markerA = {
    'sheet': 'Intensity Sum Ch=binary_mkrA Img=1',
    'columns': {
        'Intensity Sum': 'Voxel Overlap with markerA'
    }
}

# Sheet 12 --> Intensity Sum in Channel binary markerB (Ch=binary_mkrB )
markerB_markerB = {
    'sheet': 'Intensity Sum Ch=binary_mkrB Img=1',
    'columns': {
        'Intensity Sum': 'Voxel Overlap with markerB'
    }
}

# Sheet 13 --> Intensity Mean in Channel binary MV
markerB_MV_clust_vox = {
    'sheet': 'Intensity Mean Ch=binary_MV Img=1',
    'columns': {
        'Intensity Mean': 'Cluster Voxel Overlap with MV'
    }
}

# Sheet 14 --> Intensity Mean in Channel binary markerA (Ch=binary_mkrA)
markerB_markerA_clust_vox = {
    'sheet': 'Intensity Mean Ch=binary_mkrA Img=1',
    'columns': {
        'Intensity Mean': 'Cluster Voxel Overlap with markerA'
    }
}

# Sheet 15 --> get the intensity in markerA channel (Ch=mkrA )
markerB_int_mean_markerA = {
    'sheet': 'Intensity Mean Ch=mkrA Img=1',   # markerA channel --> Ch=mkrA 
    'columns': {
        'Intensity Mean': 'Intensity Mean (markerA Channel)'
    }
}

# Sheet 16 --> get the intensity sum in markerA channel (Ch=mkrA )
markerB_int_sum_markerA = {
    'sheet': 'Intensity Sum Ch=mkrA Img=1',   # markerA channel --> Ch=mkrA 
    'columns': {
        'Intensity Sum': 'Intensity Sum (markerA Channel)'
    }
}

##################### Surfaces ###################################################

# Sheet 17 --> get the overlapped volumne ratio to surfaces=markerA_abv
markerB_olvr_markerA = {
    'sheet': markerB_olvr_markerA_sheet_name['Sheet Name'].tolist()[0],       # 'Overlapped Volume Ratio to -147',
    'columns':{
        'Overlapped Volume Ratio to Surfaces': 'Overlap Volume Ratio to markerA_MV'
    }
}

# Sheet 18 --> For markerB surfaces, get the overlapped volumne ratio to surfaces=MV
markerB_olvr_MV = {
    'sheet': markerB_olvr_MV_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Overlapped Volume Ratio to Surfaces': 'Overlap Volume Ratio to MV'
    }
}

# Sheet 19 --> For markerB surfaces, get the overlapped volumne to surfaces=markerA_abv
markerB_ol_V_markerA = {
    'sheet': markerB_ol_V_markerA_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Overlapped Volume to Surfaces': 'Overlap Volume to markerA_MV'
    }
}

# Sheet 20 --> For markerB surfaces, get the overlapped volumne to surfaces=MV_select
markerB_ol_V_MV = {
    'sheet': markerB_ol_V_MV_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Overlapped Volume to Surfaces': 'Overlap Volume to MV'
    }
}

# Sheet 21 --> get the Shortest Distance to surfaces=markerA_0_708
markerB_shd_markerA = {
    'sheet': markerB_shd_markerA_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Shortest Distance to Surfaces': 'Shortest Distance to markerA_MV'
    }
}

# Sheet 22 --> get the Shortest Distance to surfaces=MV_select
markerB_shd_MV = {
    'sheet': markerB_shd_MV_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Shortest Distance to Surfaces': 'Shortest Distance to MV'
    }
}

#===========Save sheet data in a DataFrame===========
markerB_data_map = [markerB_ID, markerB_Area, markerB_Intensity_Mean, markerB_Intensity_Sum, 
                markerB_DT_max, markerB_DT_min, markerB_DT_mean, markerB_V, markerB_Vox, markerB_MV, 
                markerB_markerA, markerB_markerB, markerB_MV_clust_vox, markerB_markerA_clust_vox, markerB_int_mean_markerA, markerB_int_sum_markerA, 
                markerB_olvr_markerA, markerB_olvr_MV, markerB_ol_V_markerA, markerB_ol_V_MV, markerB_shd_markerA, markerB_shd_MV]

# DataFrame
df_markerB_All = load_xls_as_df(markerB_All, markerB_data_map)

# filter DataFrame to exclude voxels below 2
df_markerB_All = df_markerB_All[df_markerB_All['Number of Voxels'] > 2]

## 7. MV Statistics

### 7.1 Load sheet_data for MV Data (MV selection) 
* load to df_MV_selec_sheet_data
* Check for sheet name in df_MV_selec_sheet_data

In [ ]:
# File Name
MV_select = os.path.join(path, MV_data_file)
# Load data from the .xls file
MV_select_xls = xlrd.open_workbook(MV_select)
# List to store results
MV_select_sheets = []

# Iterate over all sheets
for sheet_name in MV_select_xls.sheet_names():
    sheet = MV_select_xls.sheet_by_name(sheet_name)
    
    # Safely try to get the first cell
    try:
        first_cell = sheet.cell_value(0, 0)
    except IndexError:
        first_cell = None
    
    MV_select_sheets.append({'Sheet Name': sheet_name, 'Data Name': first_cell})

# Create a DataFrame
df_MV_select_sheets = pd.DataFrame(MV_select_sheets)

# Get data for overlapped volume ratio, overlapped volume, and shortest distance to surfaces for both markerA and markerB
MV_olvr_markerA_all_sheet_name = df_MV_select_sheets[
df_MV_select_sheets['Data Name'].astype(str).str.contains('Overlapped Volume Ratio to Surfaces', na=False) &
df_MV_select_sheets['Data Name'].astype(str).str.contains('markerA', na=False) & 
~df_MV_select_sheets['Data Name'].astype(str).str.contains('Selection', na=False)
]

MV_olvr_markerA_s_sheet_name = df_MV_select_sheets[
df_MV_select_sheets['Data Name'].astype(str).str.contains('Overlapped Volume Ratio to Surfaces', na=False) &
df_MV_select_sheets['Data Name'].astype(str).str.contains('markerA', na=False) & 
df_MV_select_sheets['Data Name'].astype(str).str.contains('Selection', na=False)
]

MV_olvr_markerB_all_sheet_name = df_MV_select_sheets[
df_MV_select_sheets['Data Name'].astype(str).str.contains('Overlapped Volume Ratio to Surfaces', na=False) &
df_MV_select_sheets['Data Name'].astype(str).str.contains('markerB', na=False) & 
~df_MV_select_sheets['Data Name'].astype(str).str.contains('Selection', na=False)
]

MV_olvr_markerB_s_sheet_name = df_MV_select_sheets[
df_MV_select_sheets['Data Name'].astype(str).str.contains('Overlapped Volume Ratio to Surfaces', na=False) &
df_MV_select_sheets['Data Name'].astype(str).str.contains('markerB', na=False) & 
df_MV_select_sheets['Data Name'].astype(str).str.contains('Selection', na=False)
]

MV_ol_V_markerA_all_sheet_name = df_MV_select_sheets[
df_MV_select_sheets['Data Name'].astype(str).str.contains('Overlapped Volume to Surfaces', na=False) &
df_MV_select_sheets['Data Name'].astype(str).str.contains('markerA', na=False) & 
~df_MV_select_sheets['Data Name'].astype(str).str.contains('Selection', na=False)
]

MV_ol_V_markerA_s_sheet_name = df_MV_select_sheets[
df_MV_select_sheets['Data Name'].astype(str).str.contains('Overlapped Volume to Surfaces', na=False) &
df_MV_select_sheets['Data Name'].astype(str).str.contains('markerA', na=False) & 
df_MV_select_sheets['Data Name'].astype(str).str.contains('Selection', na=False)
]

MV_ol_V_markerB_all_sheet_name = df_MV_select_sheets[
df_MV_select_sheets['Data Name'].astype(str).str.contains('Overlapped Volume to Surfaces', na=False) &
df_MV_select_sheets['Data Name'].astype(str).str.contains('markerB', na=False) & 
~df_MV_select_sheets['Data Name'].astype(str).str.contains('Selection', na=False)
]

MV_ol_V_markerB_s_sheet_name = df_MV_select_sheets[
df_MV_select_sheets['Data Name'].astype(str).str.contains('Overlapped Volume to Surfaces', na=False) &
df_MV_select_sheets['Data Name'].astype(str).str.contains('markerB', na=False) & 
df_MV_select_sheets['Data Name'].astype(str).str.contains('Selection', na=False)
]

MV_shd_markerA_all_sheet_name = df_MV_select_sheets[
df_MV_select_sheets['Data Name'].astype(str).str.contains('Shortest Distance to Surfaces', na=False) &
df_MV_select_sheets['Data Name'].astype(str).str.contains('markerA', na=False) & 
~df_MV_select_sheets['Data Name'].astype(str).str.contains('Selection', na=False)
]

MV_shd_markerA_s_sheet_name = df_MV_select_sheets[
df_MV_select_sheets['Data Name'].astype(str).str.contains('Shortest Distance to Surfaces', na=False) &
df_MV_select_sheets['Data Name'].astype(str).str.contains('markerA', na=False) & 
df_MV_select_sheets['Data Name'].astype(str).str.contains('Selection', na=False)
]

MV_shd_markerB_all_sheet_name = df_MV_select_sheets[
df_MV_select_sheets['Data Name'].astype(str).str.contains('Shortest Distance to Surfaces', na=False) &
df_MV_select_sheets['Data Name'].astype(str).str.contains('markerB', na=False) & 
~df_MV_select_sheets['Data Name'].astype(str).str.contains('Selection', na=False)
]

MV_shd_markerB_s_sheet_name = df_MV_select_sheets[
df_MV_select_sheets['Data Name'].astype(str).str.contains('Shortest Distance to Surfaces', na=False) &
df_MV_select_sheets['Data Name'].astype(str).str.contains('markerB', na=False) & 
df_MV_select_sheets['Data Name'].astype(str).str.contains('Selection', na=False)
]

# Save in a DataFrame containing overlapped volume ratio, overlapped volume, and shortest distance to surfaces for both markerA and markerB
df_MV_selec_sheet_data = pd.concat([MV_olvr_markerA_all_sheet_name, MV_olvr_markerA_s_sheet_name, MV_olvr_markerB_all_sheet_name, MV_olvr_markerB_s_sheet_name,
                                   MV_ol_V_markerA_all_sheet_name, MV_ol_V_markerA_s_sheet_name, MV_ol_V_markerB_all_sheet_name, MV_ol_V_markerB_s_sheet_name, 
                                   MV_shd_markerA_all_sheet_name, MV_shd_markerA_s_sheet_name, MV_shd_markerB_all_sheet_name, MV_shd_markerB_s_sheet_name])

### 7.2 Load MV (MV selection) data by sheet in dataframe
* dataframe: df_MV_select

In [ ]:
# Enter Selected Sheets and Columns As Dictionaries

# Sheet 1 --> Surface ID
MV_ID = {
    'sheet': 'Area', 
    'columns':{
        'ID': 'ID'
    }
}

# Sheet 2 --> Distance Transformation max from cell core (Ch=DistTrans )
MV_DT_max = {
    'sheet': 'Intensity Max Ch=DistTrans Img=1',
    'columns':{
        'Intensity Max': 'Distance to cell (max)'
    }
} 

# Sheet 3 --> Distance Transformation max from cell core (Ch=DistTrans )
MV_DT_min = {
    'sheet': 'Intensity Min Ch=DistTrans Img=1',
    'columns':{
        'Intensity Min': 'Distance to cell (min)'
    }
} 

# Sheet 4 --> Distance Transformation max from cell core (Ch=DistTrans )
MV_DT_mean = {
    'sheet': 'Intensity Mean Ch=DistTrans Img=1',
    'columns':{
        'Intensity Mean': 'Distance to cell (mean)'
    }
} 

# Sheet 5 --> Volume
MV_V = {
    'sheet': 'Volume',
    'columns': {
        'Volume':'Volume'
    }
}

# Sheet 6 --> MV in MV
MV_in_MV = {
    'sheet': 'Intensity Sum Ch=binary_MV Img=1',
    'columns': {
        'Intensity Sum': 'MV in MV'
    }
}

# Sheet 7 --> MV: Intensity Sum in Channel binary markerA (Ch=binary_mkrA )
MV_in_markerA = {
    'sheet': 'Intensity Sum Ch=binary_mkrA Img=1',
    'columns': {
        'Intensity Sum': 'MV in markerA'
    }
}

# Sheet 8 --> Intensity Sum in Channel binary markerB (Ch=binary_mkrB )
MV_in_markerB = {
    'sheet': 'Intensity Sum Ch=binary_mkrB Img=1',
    'columns': {
        'Intensity Sum': 'MV in markerB'
    }
}

# Sheet 9 --> Intensity Mean in Channel binary markerA (Ch=binary_mkrA )
markerA_MV_clust_vox = {
    'sheet': 'Intensity Mean Ch=binary_mkrA Img=1',
    'columns': {
        'Intensity Mean': 'Cluster Voxel Overlap with markerA'
    }
}
# Sheet 10 --> Intensity Mean in Channel binary markerB (Ch=binary_mkrB )
markerB_MV_clust_vox = {
    'sheet': 'Intensity Mean Ch=binary_mkrB Img=1',
    'columns': {
        'Intensity Mean': 'Cluster Voxel Overlap with markerB'
    }
}

# Sheet 11 --> MV Ellipsoid Axis
MV_EA_A = {
    'sheet': 'Ellipsoid Axis Length A',
    'columns': {
        'Ellipsoid Axis Length A': 'Ellipsoid Axis Length A'
    }
}

# Sheet 12 --> MV Ellipsoid Axis
MV_EA_B = {
    'sheet': 'Ellipsoid Axis Length B',
    'columns': {
        'Ellipsoid Axis Length B': 'Ellipsoid Axis Length B'
    }
}

# Sheet 13 --> MV Ellipsoid Axis
MV_EA_C = {
    'sheet': 'Ellipsoid Axis Length C',
    'columns': {
        'Ellipsoid Axis Length C': 'Ellipsoid Axis Length C'
    }
}

# Sheet 14 --> MV Ellipticity (oblate)
MV_E_oblate = {
    'sheet': 'Ellipticity (oblate)',
    'columns': {
        'Ellipticity (oblate)': 'Ellipticity (oblate)'
    }
}

# Sheet 15 --> MV Ellipticity (prolate)
MV_E_prolate = {
    'sheet': 'Ellipticity (prolate)',
    'columns': {
        'Ellipticity (prolate)': 'Ellipticity (prolate)'
    }
}

################ Surfaces#############################

## Overlapped volumn ratio to surfaces

# Sheet 16 --> get the overlapped volumne ratio to surfaces=markerA_all
MV_olvr_markerA_all = {
    'sheet': MV_olvr_markerA_all_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Overlapped Volume Ratio to Surfaces': 'Overlap Volume Ratio to markerA_all'
    }
}

# Sheet 17 --> get the overlapped volumne ratio to surfaces=markerA (selection abv threshold)
MV_olvr_markerA_s = {
    'sheet': MV_olvr_markerA_s_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Overlapped Volume Ratio to Surfaces': 'Overlap Volume Ratio to markerA_MV'
    }
}

# Sheet 18 --> get the overlapped volumne ratio to surfaces=markerB_all
MV_olvr_markerB_all = {
    'sheet': MV_olvr_markerB_all_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Overlapped Volume Ratio to Surfaces': 'Overlap Volume Ratio to markerB_all'
    }
}

# Sheet 19 --> get the overlapped volumne ratio to surfaces=markerB (selection abv threshold)
MV_olvr_markerB_s = {
    'sheet': MV_olvr_markerB_s_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Overlapped Volume Ratio to Surfaces': 'Overlap Volume Ratio to markerB_MV'
    }
}

## Overlapped Volumn to Surfaces

# Sheet 20 --> For MV surfaces, get the overlapped volumne to surfaces=markerA_all
MV_ol_V_markerA_all = {
    'sheet': MV_ol_V_markerA_all_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Overlapped Volume to Surfaces': 'Overlap Volume to markerA_all'
    }
}

# Sheet 21 --> For MV surfaces, get the overlapped volumne to surfaces=markerA (selection abv threshold)
MV_ol_V_markerA_s = {
    'sheet': MV_ol_V_markerA_s_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Overlapped Volume to Surfaces': 'Overlap Volume to markerA_s'
    }
}

# Sheet 22 --> For MV surfaces, get the overlapped volumne to surfaces=markerB_all
MV_ol_V_markerB_all = {
    'sheet': MV_ol_V_markerB_all_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Overlapped Volume to Surfaces': 'Overlap Volume to markerB_all'
    }
}

# Sheet 23 --> For MV surfaces, get the overlapped volumne to surfaces=markerB (selection abv threshold)
MV_ol_V_markerB_s = {
    'sheet': MV_ol_V_markerB_s_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Overlapped Volume to Surfaces': 'Overlap Volume to markerB_s'
    }
}

## =============Shortest Distance to Surfaces===========================================

# Sheet 24 --> get the Shortest Distance to surfaces=markerA_all
MV_shd_markerA_all = {
    'sheet': MV_shd_markerA_all_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Shortest Distance to Surfaces': 'Shortest Distance to markerA_all'
    }
}

# Sheet 25 --> get the Shortest Distance to surfaces=markerA_s (selection abv threshold)
MV_shd_markerA_s = {
    'sheet': MV_shd_markerA_s_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Shortest Distance to Surfaces': 'Shortest Distance to markerA_s'
    }
}

# Sheet 26 --> get the Shortest Distance to surfaces=markerB_all
MV_shd_markerB_all = {
    'sheet': MV_shd_markerB_all_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Shortest Distance to Surfaces': 'Shortest Distance to markerB_all'
    }
}

# Sheet 27 --> get the Shortest Distance to surfaces=markerB_s (selection abv threshold)
MV_shd_markerB_s = {
    'sheet': MV_shd_markerB_s_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Shortest Distance to Surfaces': 'Shortest Distance to markerB_s'
    }
}

# =====================Save sheet data in a DataFrame========================

MV_data_map = [MV_ID, MV_DT_max, MV_DT_min, MV_DT_mean, MV_V,                    # Volume and Intensity
               MV_EA_A, MV_EA_B, MV_EA_C, MV_E_oblate, MV_E_prolate, # Shape
               MV_in_markerA, MV_in_markerB, MV_in_MV, markerA_MV_clust_vox, markerB_MV_clust_vox, # Overlap with binary_markerA and binary_markerB
               MV_olvr_markerA_all, MV_olvr_markerA_s, MV_olvr_markerB_all, MV_olvr_markerB_s, # Overlapped Volumn Ratio with surfaces
               MV_ol_V_markerA_all, MV_ol_V_markerA_s, MV_ol_V_markerB_all, MV_ol_V_markerB_s, # Overlapped Volume with surfaces
               MV_shd_markerA_all, MV_shd_markerA_s, MV_shd_markerB_all, MV_shd_markerB_s # Shortes distance to surfaces
              ]

# DataFrame
df_MV_select = load_xls_as_df(MV_select, MV_data_map)

## 8. markerA statistics for clusters in selection

### 8.1 Load data for markerA surfaces: selection 
* Save in df_markerA_selec_sheet_data
* Check for sheet name in df_markerA_selec_sheet_data

In [ ]:
# Enter File Name
markerA_selec = os.path.join(path, markerA_select_data_file)
# Load data from the .xls file
markerA_selec_xls = xlrd.open_workbook(markerA_selec)
# List to store results
markerA_selec_sheets = []

# Iterate over all sheets
for sheet_name in markerA_selec_xls.sheet_names():
    sheet = markerA_selec_xls.sheet_by_name(sheet_name)
    
    # Safely try to get the first cell
    try:
        first_cell = sheet.cell_value(0, 0)
    except IndexError:
        first_cell = None
    
    markerA_selec_sheets.append({'Sheet Name': sheet_name, 'Data Name': first_cell})

# Create a DataFrame
df_markerA_selec_sheets = pd.DataFrame(markerA_selec_sheets)

# Find which sheet to read based on the data-content
markerA_olvr_markerB_sheet_name = df_markerA_selec_sheets[
df_markerA_selec_sheets['Data Name'].astype(str).str.contains('Overlapped Volume Ratio to Surfaces', na=False) &
df_markerA_selec_sheets['Data Name'].astype(str).str.contains('markerB', na=False) & 
df_markerA_selec_sheets['Data Name'].astype(str).str.contains('Selection', na=False)
]

# Find which sheet to read based on the data-content
markerA_olvr_MV_sheet_name = df_markerA_selec_sheets[
df_markerA_selec_sheets['Data Name'].astype(str).str.contains('Overlapped Volume Ratio to Surfaces', na=False) &
df_markerA_selec_sheets['Data Name'].astype(str).str.contains('MV', na=False)
]

# Find which sheet to read based on the data-content
markerA_ol_V_markerB_sheet_name = df_markerA_selec_sheets[
df_markerA_selec_sheets['Data Name'].astype(str).str.contains('Overlapped Volume to Surfac', na=False) &
df_markerA_selec_sheets['Data Name'].astype(str).str.contains('markerB', na=False) & 
df_markerA_selec_sheets['Data Name'].astype(str).str.contains('Selection', na=False)
]

# Find which sheet to read based on the data-content
markerA_ol_V_MV_sheet_name = df_markerA_selec_sheets[
df_markerA_selec_sheets['Data Name'].astype(str).str.contains('Overlapped Volume to Surfaces', na=False) &
df_markerA_selec_sheets['Data Name'].astype(str).str.contains('MV', na=False)
]

markerA_shd_markerB_sheet_name = df_markerA_selec_sheets[
df_markerA_selec_sheets['Data Name'].astype(str).str.contains('Shortest Distance to Surfaces', na=False) &
df_markerA_selec_sheets['Data Name'].astype(str).str.contains('markerB', na=False) & 
df_markerA_selec_sheets['Data Name'].astype(str).str.contains('Selection', na=False)
]

markerA_shd_MV_sheet_name = df_markerA_selec_sheets[
df_markerA_selec_sheets['Data Name'].astype(str).str.contains('Shortest Distance to Surfaces', na=False) &
df_markerA_selec_sheets['Data Name'].astype(str).str.contains('MV', na=False)
]

# Save in a DataFrame containing overlapped volume ratio, overlapped volume, and shortest distance to surfaces for both markerB and MV
df_markerA_selec_sheet_data = pd.concat([markerA_olvr_markerB_sheet_name, markerA_olvr_MV_sheet_name,
                                    markerA_ol_V_markerB_sheet_name, markerA_ol_V_MV_sheet_name,
                                    markerA_shd_markerB_sheet_name, markerA_shd_MV_sheet_name])

### 8.2 Load data for markerA selection by sheet as dataframe
* Save as df_markerA_selec

In [ ]:
# Enter Selected Sheets and Columns As Dictionaries

# Sheet 1 --> Surface ID
markerA_ID ={
    'sheet': 'Area', 
    'columns':{
        'ID': 'ID'
    }
}

# Sheet 2 --> Area
markerA_Area ={
    'sheet': 'Area', 
    'columns':{
        'Area': 'Area'
    }
}

# Sheet 3 --> markerA Intensity Mean (markerA channel --> Ch=mkrA )
markerA_Intensity_Mean = {
    'sheet': 'Intensity Mean Ch=mkrA Img=1',   # markerA channel --> Ch=mkrA 
    'columns': {
        'Intensity Mean': 'Intensity Mean markerA'
    }
}

# Sheet 4 --> markerA Intensity Mean (markerA channel --> Ch=mkrA )
markerA_Intensity_Sum = {
    'sheet': 'Intensity Sum Ch=mkrA Img=1',   # markerA channel --> Ch=mkrA 
    'columns': {
        'Intensity Sum': 'Intensity Sum markerA'
    }
}

# Sheet 5 --> Distance Transformation from cell core (Distance Transformation channel --> Ch=DistTrans )
markerA_DT_max = {
    'sheet': 'Intensity Max Ch=DistTrans Img=1',  # channel --> Ch=DistTrans 
    'columns':{
        'Intensity Max': 'Distance to cell (max)'
    }
}

# Sheet 6 --> Distance Transformation from cell core (Distance Transformation channel --> Ch=DistTrans )
markerA_DT_min = {
    'sheet': 'Intensity Min Ch=DistTrans Img=1',  # channel --> Ch=DistTrans 
    'columns':{
        'Intensity Min': 'Distance to cell (min)'
    }
}

# Sheet 7
markerA_DT_mean = {
    'sheet': 'Intensity Mean Ch=DistTrans Img=1',   # channel --> Ch=DistTrans 
    'columns':{
        'Intensity Mean': 'Distance to cell (mean)'
    }
}

# Sheet 8 --> Volume
markerA_V = {
    'sheet': 'Volume',
    'columns': {
        'Volume':'Volume'
    }
}

# Sheet 9 --> Voxel Size
markerA_Vox = {
    'sheet': 'Number of Voxels Img=1',
    'columns': {
        'Number of Voxels':'Number of Voxels'
    }
}

# Sheet 10 --> Intensity Sum in Channel binary MV (channel binary MV --> Ch=binary_MV )
markerA_MV = {
    'sheet': 'Intensity Sum Ch=binary_MV Img=1',
    'columns': {
        'Intensity Sum': 'Voxel Overlap with MV'
    }
}

# Sheets 11 --> Intensity Sum in Channel binary markerB (Ch=binary_mkrB )
markerA_markerB = {
    'sheet': 'Intensity Sum Ch=binary_mkrB Img=1',
    'columns': {
        'Intensity Sum': 'Voxel Overlap with markerB'
    }
}
# Sheet 12 --> Intensity Sum in binary markerA (Ch=binary_mkrA)
markerA_markerA = {
    'sheet': 'Intensity Sum Ch=binary_mkrA Img=1',
    'columns': {
        'Intensity Sum': 'Voxel Overlap with markerA'
    }
}

# Sheet 13 --> Intensity Mean in Channel binary markerB (Ch=binary_mkrB)
markerA_markerB_clust_vox = {
    'sheet': 'Intensity Mean Ch=binary_mkrB Img=1',
    'columns': {
        'Intensity Mean': 'Cluster Voxel Overlap with markerB'
    }
}

# Sheet 14 --> get the intensity in markerB channel (Ch=mkrB )
markerA_int_mean_markerB = {
    'sheet': 'Intensity Mean Ch=mkrB Img=1',   # markerB channel --> Ch=mkrB 
    'columns': {
        'Intensity Mean': 'Intensity Mean (markerB Channel)'
    }
}

# Sheet 15 --> get the intensity sum in markerB channel (Ch=mkrB )
markerA_int_sum_markerB = {
    'sheet': 'Intensity Sum Ch=mkrB Img=1',   # markerB channel --> Ch=mkrB 
    'columns': {
        'Intensity Sum': 'Intensity Sum (markerB Channel)'
    }
}

##################### Surfaces ###################################################

# Sheet 16 --> get the overlapped volumne ratio to surfaces=markerB_abv
markerA_olvr_markerB = {
    'sheet': markerA_olvr_markerB_sheet_name['Sheet Name'].tolist()[0],       # 'Overlapped Volume Ratio to -147',
    'columns':{
        'Overlapped Volume Ratio to Surfaces': 'Overlap Volume Ratio to markerB_MV'
    }
}

# Sheet 17 --> For markerA surfaces, get the overlapped volumne ratio to surfaces=MV
markerA_olvr_MV = {
    'sheet': markerA_olvr_MV_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Overlapped Volume Ratio to Surfaces': 'Overlap Volume Ratio to MV'
    }
}

# Sheet 18 --> For markerA surfaces, get the overlapped volumne to surfaces=markerB_abv
markerA_ol_V_markerB = {
    'sheet': markerA_ol_V_markerB_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Overlapped Volume to Surfaces': 'Overlap Volume to markerB_MV'
    }
}

# Sheet 19 --> For markerA surfaces, get the overlapped volumne to surfaces=MV_select
markerA_ol_V_MV = {
    'sheet': markerA_ol_V_MV_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Overlapped Volume to Surfaces': 'Overlap Volume to MV'
    }
}

# Sheet 20 --> get the Shortest Distance to surfaces=markerB_0_708
markerA_shd_markerB = {
    'sheet': markerA_shd_markerB_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Shortest Distance to Surfaces': 'Shortest Distance to markerB_MV'
    }
}

# Sheet 21 --> get the Shortest Distance to surfaces=MV_select
markerA_shd_MV = {
    'sheet': markerA_shd_MV_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Shortest Distance to Surfaces': 'Shortest Distance to MV'
    }
}

# ====================Save sheet data in a DataFrame========================

markerA_selec_data_map = [markerA_ID, markerA_Area, markerA_Intensity_Mean, markerA_Intensity_Sum, markerA_DT_max, markerA_DT_min, 
                markerA_DT_mean, markerA_V, markerA_Vox, markerA_MV, markerA_markerB, markerA_markerA, markerA_markerB_clust_vox,
                markerA_int_mean_markerB, markerA_int_sum_markerB, markerA_olvr_markerB, markerA_olvr_MV, markerA_ol_V_markerB, 
                markerA_ol_V_MV, markerA_shd_markerB, markerA_shd_MV]

# DataFrame
df_markerA_selec = load_xls_as_df(markerA_selec, markerA_selec_data_map)

# filter DataFrame to exclude voxels below 2
df_markerA_selec = df_markerA_selec[df_markerA_selec['Number of Voxels'] > 2]

## 9. markerB statistics for clusters in selection

### 9.1 Load markerB surfaces: selection as dataframe
* Save as df_markerB_selec_sheet_data
* Check for sheet name in df_markerB_selec_sheet_data

In [ ]:
# File Name
markerB_selec = os.path.join(path, markerB_select_data_file)
# Load data from the .xls file
markerB_selec_xls = xlrd.open_workbook(markerB_selec)

# List to store results
markerB_selec_sheets = []

# Iterate over all sheets
for sheet_name in markerB_selec_xls.sheet_names():
    sheet = markerB_selec_xls.sheet_by_name(sheet_name)
    
    # Safely try to get the first cell
    try:
        first_cell = sheet.cell_value(0, 0)
    except IndexError:
        first_cell = None
    
    markerB_selec_sheets.append({'Sheet Name': sheet_name, 'Data Name': first_cell})

# Create a DataFrame
df_markerB_selec_sheets = pd.DataFrame(markerB_selec_sheets)
# Find which sheet to read based on the data-content
markerB_olvr_markerA_sheet_name = df_markerB_selec_sheets[
df_markerB_selec_sheets['Data Name'].astype(str).str.contains('Overlapped Volume Ratio to Surfaces', na=False) &
df_markerB_selec_sheets['Data Name'].astype(str).str.contains('markerA', na=False) & 
df_markerB_selec_sheets['Data Name'].astype(str).str.contains('Selection', na=False)
]

# Find which sheet to read based on the data-content
markerB_olvr_MV_sheet_name = df_markerB_selec_sheets[
df_markerB_selec_sheets['Data Name'].astype(str).str.contains('Overlapped Volume Ratio to Surfaces', na=False) &
df_markerB_selec_sheets['Data Name'].astype(str).str.contains('MV', na=False)
]

# Find which sheet to read based on the data-content
markerB_ol_V_markerA_sheet_name = df_markerB_selec_sheets[
df_markerB_selec_sheets['Data Name'].astype(str).str.contains('Overlapped Volume to Surfac', na=False) &
df_markerB_selec_sheets['Data Name'].astype(str).str.contains('markerA', na=False) & 
df_markerB_selec_sheets['Data Name'].astype(str).str.contains('Selection', na=False)
]

# Find which sheet to read based on the data-content
markerB_ol_V_MV_sheet_name = df_markerB_selec_sheets[
df_markerB_selec_sheets['Data Name'].astype(str).str.contains('Overlapped Volume to Surfaces', na=False) &
df_markerB_selec_sheets['Data Name'].astype(str).str.contains('MV', na=False)
]

markerB_shd_markerA_sheet_name = df_markerB_selec_sheets[
df_markerB_selec_sheets['Data Name'].astype(str).str.contains('Shortest Distance to Surfaces', na=False) &
df_markerB_selec_sheets['Data Name'].astype(str).str.contains('markerA', na=False) & 
df_markerB_selec_sheets['Data Name'].astype(str).str.contains('Selection', na=False)
]

markerB_shd_MV_sheet_name = df_markerB_selec_sheets[
df_markerB_selec_sheets['Data Name'].astype(str).str.contains('Shortest Distance to Surfaces', na=False) &
df_markerB_selec_sheets['Data Name'].astype(str).str.contains('MV', na=False)
]

# Save in a DataFrame containing overlapped volume ratio, overlapped volume, and shortest distance to surfaces for both markerA and MV
df_markerB_selec_sheet_data = pd.concat([markerB_olvr_markerA_sheet_name, markerB_olvr_MV_sheet_name, 
                                      markerB_ol_V_markerA_sheet_name, markerB_ol_V_MV_sheet_name, 
                                      markerB_shd_markerA_sheet_name, markerB_shd_MV_sheet_name])

### 9.2 Load markerB selection by sheet as dataframe
* Save in df_markerB_selec

In [ ]:
# Enter Selected Sheets and Columns As Dictionaries

# Sheet 1 --> Surface ID
markerB_ID ={
    'sheet': 'Area', 
    'columns':{
        'ID': 'ID'
    }
}

# Sheet 2 --> Area
markerB_Area ={
    'sheet': 'Area', 
    'columns':{
        'Area': 'Area'
    }
}

# Sheet 3 --> markerB Intensity Mean (markerB channel --> Ch=mkrB )
markerB_Intensity_Mean = {
    'sheet': 'Intensity Mean Ch=mkrB Img=1',   # markerB channel --> Ch=mkrB 
    'columns': {
        'Intensity Mean': 'Intensity Mean markerB'
    }
}

# Sheet 4 --> markerB Intensity Mean (markerB channel --> Ch=mkrB )
markerB_Intensity_Sum = {
    'sheet': 'Intensity Sum Ch=mkrB Img=1',   # markerB channel --> Ch=mkrB 
    'columns': {
        'Intensity Sum': 'Intensity Sum markerB'
    }
}

# Sheet 5 --> Distance Transformation from cell core (Distance Transformation channel --> Ch=DistTrans )
markerB_DT_max = {
    'sheet': 'Intensity Max Ch=DistTrans Img=1',  # channel --> Ch=DistTrans 
    'columns':{
        'Intensity Max': 'Distance to cell (max)'
    }
}


# Sheet 6 --> Distance Transformation from cell core (Distance Transformation channel --> Ch=DistTrans )
markerB_DT_min = {
    'sheet': 'Intensity Min Ch=DistTrans Img=1',  # channel --> Ch=DistTrans 
    'columns':{
        'Intensity Min': 'Distance to cell (min)'
    }
}

# Sheet 7
markerB_DT_mean = {
    'sheet': 'Intensity Mean Ch=DistTrans Img=1',   # channel --> Ch=DistTrans 
    'columns':{
        'Intensity Mean': 'Distance to cell (mean)'
    }
}

# Sheet 8 --> Volume
markerB_V = {
    'sheet': 'Volume',
    'columns': {
        'Volume':'Volume'
    }
}

# Sheet 9 --> Voxel Size
markerB_Vox = {
    'sheet': 'Number of Voxels Img=1',
    'columns': {
        'Number of Voxels':'Number of Voxels'
    }
}

# Sheet 10 --> Intensity Sum in Channel binary MV (channel binary MV --> Ch=binary_MV )
markerB_MV = {
    'sheet': 'Intensity Sum Ch=binary_MV Img=1',
    'columns': {
        'Intensity Sum': 'Voxel Overlap with MV'
    }
}

# Sheets 11 --> Intensity Sum in Channel binary markerA (Ch=binary_mkrA )
markerB_markerA = {
    'sheet': 'Intensity Sum Ch=binary_mkrA Img=1',
    'columns': {
        'Intensity Sum': 'Voxel Overlap with markerA'
    }
}
# Sheet 12 --> Intensity Sum in Channel binary markerB (Ch=binary_mkrB )
markerB_markerB = {
    'sheet': 'Intensity Sum Ch=binary_mkrB Img=1',
    'columns': {
        'Intensity Sum': 'Voxel Overlap with markerB'
    }
}

# Sheet 13 --> Intensity Mean in Channel binary markerA (Ch=binary_mkrA)
markerB_markerA_clust_vox = {
    'sheet': 'Intensity Mean Ch=binary_mkrA Img=1',
    'columns': {
        'Intensity Mean': 'Cluster Voxel Overlap with markerA'
    }
}

# Sheet 14 --> get the intensity in markerA channel (Ch=mkrA )
markerB_int_mean_markerA = {
    'sheet': 'Intensity Mean Ch=mkrA Img=1',   # markerA channel --> Ch=mkrA 
    'columns': {
        'Intensity Mean': 'Intensity Mean (markerA Channel)'
    }
}

# Sheet 15 --> get the intensity sum in markerA channel (Ch=1)
markerB_int_sum_markerA = {
    'sheet': 'Intensity Sum Ch=mkrA Img=1',   # markerA channel --> Ch=1
    'columns': {
        'Intensity Sum': 'Intensity Sum (markerA Channel)'
    }
}

##################### Surfaces ###################################################

# Sheet 16 --> get the overlapped volumne ratio to surfaces=markerA_abv
markerB_olvr_markerA = {
    'sheet': markerB_olvr_markerA_sheet_name['Sheet Name'].tolist()[0],       # 'Overlapped Volume Ratio to -147',
    'columns':{
        'Overlapped Volume Ratio to Surfaces': 'Overlap Volume Ratio to markerA_MV'
    }
}

# Sheet 17 --> For markerB surfaces, get the overlapped volumne ratio to surfaces=MV
markerB_olvr_MV = {
    'sheet': markerB_olvr_MV_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Overlapped Volume Ratio to Surfaces': 'Overlap Volume Ratio to MV'
    }
}

# Sheet 18 --> For markerB surfaces, get the overlapped volumne to surfaces=markerA_abv
markerB_ol_V_markerA = {
    'sheet': markerB_ol_V_markerA_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Overlapped Volume to Surfaces': 'Overlap Volume to markerA_MV'
    }
}

# Sheet 19 --> For markerB surfaces, get the overlapped volumne to surfaces=MV_select
markerB_ol_V_MV = {
    'sheet': markerB_ol_V_MV_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Overlapped Volume to Surfaces': 'Overlap Volume to MV'
    }
}

# Sheet 20 --> get the Shortest Distance to surfaces=markerA_0_708
markerB_shd_markerA = {
    'sheet': markerB_shd_markerA_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Shortest Distance to Surfaces': 'Shortest Distance to markerA_MV'
    }
}

# Sheet 21 --> get the Shortest Distance to surfaces=MV_select
markerB_shd_MV = {
    'sheet': markerB_shd_MV_sheet_name['Sheet Name'].tolist()[0],
    'columns':{
        'Shortest Distance to Surfaces': 'Shortest Distance to MV'
    }
}
markerB_selec_data_map = [markerB_ID, markerB_Area, markerB_Intensity_Mean, markerB_Intensity_Sum, markerB_DT_max, markerB_DT_min, markerB_DT_mean, 
                markerB_V, markerB_Vox, markerB_MV, markerB_markerA, markerB_markerB, markerB_markerA_clust_vox, 
                markerB_int_mean_markerA, markerB_int_sum_markerA, markerB_olvr_markerA, markerB_olvr_MV, markerB_ol_V_markerA, markerB_ol_V_MV, 
                markerB_shd_markerA, markerB_shd_MV]

# ===========================Save in DataFrame=========================
df_markerB_selec = load_xls_as_df(markerB_selec, markerB_selec_data_map)
# filter DataFrame to exclude voxels below 2
df_markerB_selec = df_markerB_selec[df_markerB_selec['Number of Voxels'] > 2]

## 10. Save dataframes from 5-9 as Excel file
* Output Excel files:
  * df_data_loaded.xlsx
    * Dataframe loaded: 
      * df_markerA_All
      * df_markerB_All
      * df_MV_select
      * df_markerA_selec
      * df_markerB_selec
  * check_sheet_name.xlsx
    * Dataframe loaded:
      * df_markerA_All_sheet_data
      * df_markerB_All_sheet_data
      * df_MV_select_sheet_data
      * df_markerA_selec_sheet_data
      * df_markerB_selec_sheet_data

In [ ]:
# Save all Receptor and MV surface from dataframes in an Excel file with multiple sheets
with pd.ExcelWriter(os.path.join(result_folder_path, 'df_data_loaded.xlsx'), engine='xlsxwriter') as writer:
    df_markerA_All.to_excel(writer, sheet_name='df_markerA_All', index=False)
    df_markerB_All.to_excel(writer, sheet_name='df_markerB_All', index=False)
    df_MV_select.to_excel(writer, sheet_name='df_MV_select', index=False)
    df_markerA_selec.to_excel(writer, sheet_name='df_markerA_selec', index=False)
    df_markerB_selec.to_excel(writer, sheet_name='df_markerB_selec', index=False)

# Save all the sheet names of Receptor and MV surface from dataframes in an Excel file with multiple sheets
with pd.ExcelWriter(os.path.join(result_folder_path, 'check_sheet_name.xlsx'), engine='xlsxwriter') as writer:
    df_markerA_all_sheet_data.to_excel(writer, sheet_name='markerA_all', index=False)
    df_markerB_all_sheet_data.to_excel(writer, sheet_name='markerB_all', index=False)
    df_MV_selec_sheet_data.to_excel(writer, sheet_name='MV_selec', index=False)
    df_markerA_selec_sheet_data.to_excel(writer, sheet_name='markerA_selec', index=False)
    df_markerB_selec_sheet_data.to_excel(writer, sheet_name='markerB_selec', index=False)

## 11. MV measurements
* MV size:
  * Volume
  * Area
  * Length 
  * Diameter 

In [ ]:
# Volume
mVol_MV = df_MV_select['Volume'].mean()
print(mVol_MV)

# Plot
sns.kdeplot(df_MV_select['Volume'])
plt.xlabel('Volume')
plt.ylabel('Density')

In [ ]:
# Load data for all clusters

# What data to load
data_1 = 'Ellipsoid Axis Length A'
data_2 = 'Ellipsoid Axis Length B'
data_3 = 'Ellipsoid Axis Length C'
data_4_1 = 'Distance to cell (max)'
data_4_2 = 'Distance to cell (min)'


# Shortest Distance to MV
group1 = df_MV_select[data_1]
group2 = df_MV_select[data_2]
group3 = df_MV_select[data_3]
group4 = df_MV_select[data_4_1] - df_MV_select[data_4_2]
# Combine data into a DataFrame
df1 = pd.DataFrame({
    'value': np.concatenate([group1, group2, group3, group4]),
    'group': ['Axis A']*len(group1) + ['Axis B']*len(group2) + ['Axis C']*len(group3) + ['Delta DT']*len(group4)
})

## 12. Distributions of cluster distance from the membrane
* Use "Distance Transformation Channel Intensity" to determine the furthest point of the cluster distance from membrane

In [ ]:
group1 = df_markerA_All['Distance to cell (max)']
group2 = df_markerB_All['Distance to cell (max)']
group3 = df_markerA_selec['Distance to cell (max)']
group4 = df_markerB_selec['Distance to cell (max)']
group5 = df_MV_select['Distance to cell (max)']

df_data = pd.DataFrame({
    'group': ['markerA all']*len(group1) + ['markerB all']*len(group2) + ['markerA MV']*len(group3) + ['markerB MV']*len(group4) + ['MV']*len(group5),
    'value': np.concatenate([group1, group2, group3, group4, group5])
})

plt.figure(figsize=(8, 6))

for group in df_data['group'].unique():
    subset = df_data[df_data['group'] == group]
    sns.kdeplot(data=subset, x='value', label=f'{group}')

plt.legend(title='Group')
plt.xlabel('Data Values')
plt.ylabel('Density')
plt.title('Distance to cell (max)')
#plt.xlim(-0.1, 2)
#plt.ylim(-0.1, 6.1)

## 13. Recepors in MV

In [ ]:
# Parameters: 
#    distance_max --> surfaces need to have distance transformation > distance_max to considered in MV
#    MV_masked_sum --> surfaces need to have intensity sum in the masked_MV channel > MV_masked_sum to be considered in MV
distance_max = 0.2
MV_masked_sum = 0.9
markerA_masked_sum = 0.9
markerB_masked_sum = 0.9

### 13.1 markerA in MV

In [ ]:
# Select markerA in MV from df_markerA
df_markerA_MV = df_markerA_All[(df_markerA_All['Distance to cell (max)'] > distance_max) & (df_markerA_All['Voxel Overlap with MV'] > MV_masked_sum)]

##### Percentage in MV

In [ ]:
# Percent of markerA clusters in MV
markerA_clusters_in_MV = df_markerA_MV.shape[0]/df_markerA_All.shape[0]
print(markerA_clusters_in_MV)

In [ ]:
# Percent of markerA intensity sum in MV
markerA_int_sum_in_MV = sum(df_markerA_MV['Intensity Sum markerA']) / sum(df_markerA_All['Intensity Sum markerA'])
print(markerA_int_sum_in_MV)

In [ ]:
# Percent of markerA volume in MV
markerA_volume_in_MV = sum(df_markerA_MV['Volume']) / sum(df_markerA_All['Volume'])
print(markerA_volume_in_MV)

### 13.2 markerB in MV

In [ ]:
# Select markerB in MV from df_markerB
df_markerB_MV = df_markerB_All[(df_markerB_All['Distance to cell (max)'] > distance_max) & (df_markerB_All['Voxel Overlap with MV'] > MV_masked_sum)]

##### Percentage in MV

In [ ]:
# Percent of markerB clusters in MV
markerB_clusters_in_MV = df_markerB_MV.shape[0]/df_markerB_All.shape[0]
print(markerB_clusters_in_MV)

In [ ]:
# Percent of markerB intensity sum in MV
markerB_int_sum_in_MV = sum(df_markerB_MV['Intensity Sum markerB']) / sum(df_markerB_All['Intensity Sum markerB'])
print(markerB_int_sum_in_MV)

In [ ]:
# Percent of markerB volume in MV
markerB_volume_in_MV = sum(df_markerB_MV['Volume']) / sum(df_markerB_All['Volume'])
print(markerB_volume_in_MV)

## 14. MV with markerA / with markerB

### 14.1 Create dataframe for markerA+ MVs

In [ ]:
# Select MV with markerA +
df_MV_with_markerA = df_MV_select[df_MV_select['MV in markerA'] > markerA_masked_sum]
#print(df_MV_with_markerA)

#### 14.1.1 Percentage of markerA+ MV

In [ ]:
# Percent of markerA+ MV
percent_MV_with_markerA = df_MV_with_markerA.shape[0]/df_MV_select.shape[0]
print(percent_MV_with_markerA)

In [ ]:
# markerA volume in MV
markerA_volume_sum_in_MV = sum(df_MV_with_markerA['MV in markerA'])
print(markerA_volume_sum_in_MV)

### 14.2 Create dataframe for markerB+ MVs

In [ ]:
# Select MV with markerB+
df_MV_with_markerB = df_MV_select[df_MV_select['MV in markerB'] > markerB_masked_sum]
#print(df_MV_with_markerB)

#### 14.2.1 Percentage of markerB+ MVs

In [ ]:
# Percent of markerB+ MV
percent_MV_with_markerB = df_MV_with_markerB.shape[0]/df_MV_select.shape[0]
print(percent_MV_with_markerB)

In [ ]:
# markerB volume in MV
markerB_volume_sum_in_MV = sum(df_MV_with_markerB['MV in markerB'])
print(markerB_volume_sum_in_MV)

### 14.3 Create dataframe for MV with both markerA and markerB

In [ ]:
# Select MV with markerA and markerB+
df_MV_with_markerA_markerB = df_MV_select[(df_MV_select['MV in markerB'] > markerB_masked_sum) & (df_MV_select['MV in markerA'] > markerA_masked_sum)]
#print(df_MV_with_markerA_markerB)

#### 14.3.1 Percentage of markerA+ and markerB+ MVs

In [ ]:
# Percent markerB+ and markerA+ MV
percent_MV_with_markerA_markerB = df_MV_with_markerA_markerB.shape[0]/df_MV_select.shape[0]
print(percent_MV_with_markerA_markerB)

## 15. Colocalization Plots for Receptor Vs. MV
* Use "Overlapped Volume Ratio to Surface" and "Shortest Distance to Surface" as parameters

### 15.1 markerA profile

In [ ]:
# Create side-by-side subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# First plot: Shortest Distance to markerB_MV
sns.kdeplot(df_markerA_selec['Shortest Distance to markerB_MV'], ax=ax1)
ax1.set_xlabel('Data Values')
ax1.set_ylabel('Density')
ax1.set_title('Shortest Distance to markerB_MV')

# Second plot: Overlap Volume Ratio to markerB_MV
sns.kdeplot(df_markerA_selec['Overlap Volume Ratio to markerB_MV'], ax=ax2)
ax2.set_xlabel('Data Values')
ax2.set_ylabel('Density')
ax2.set_title('Overlap Volume Ratio to markerB_MV')

# Adjust layout
plt.tight_layout()
plt.show()

### 15.2 markerA Vs MV

In [ ]:
# Create side-by-side subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# First plot: Shortest Distance to markerB_MV
sns.kdeplot(df_markerA_selec['Shortest Distance to MV'], ax=ax1)
ax1.set_xlabel('Data Values')
ax1.set_ylabel('Density')
ax1.set_title('Shortest Distance to MV')

# Second plot: Overlap Volume Ratio to markerB_MV
sns.kdeplot(df_markerA_selec['Overlap Volume Ratio to MV'], ax=ax2)
ax2.set_xlabel('Data Values')
ax2.set_ylabel('Density')
ax2.set_title('Overlap Volume Ratio to MV')

# Adjust layout
plt.tight_layout()
plt.show()

### 15.3 markerB profile

In [ ]:
# Create side-by-side subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# First plot: Shortest Distance to markerB_MV
sns.kdeplot(df_markerB_selec['Shortest Distance to markerA_MV'], ax=ax1)
ax1.set_xlabel('Data Values')
ax1.set_ylabel('Density')
ax1.set_title('Shortest Distance to markerA_MV')

# Second plot: Overlap Volume Ratio to markerB_MV
sns.kdeplot(df_markerB_selec['Overlap Volume Ratio to markerA_MV'], ax=ax2)
ax2.set_xlabel('Data Values')
ax2.set_ylabel('Density')
ax2.set_title('Overlap Volume Ratio to markerA_MV')

# Adjust layout
plt.tight_layout()
plt.show()

### 15.4 markerB Vs MV

In [ ]:
# Create side-by-side subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# First plot: Shortest Distance to markerB_MV
sns.kdeplot(df_markerB_selec['Shortest Distance to MV'], ax=ax1)
ax1.set_xlabel('Data Values')
ax1.set_ylabel('Density')
ax1.set_title('Shortest Distance to MV')

# Second plot: Overlap Volume Ratio to markerB_MV
sns.kdeplot(df_markerB_selec['Overlap Volume Ratio to MV'], ax=ax2)
ax2.set_xlabel('Data Values')
ax2.set_ylabel('Density')
ax2.set_title('Overlap Volume Ratio to MV')

# Adjust layout
plt.tight_layout()
plt.show()

### 15.5 2D KDE plot
* Does not tell anything about population in MV VS all

In [ ]:
# Subplot 2x2
markerB_x_label = 'Shortest Distance to MV'
markerB_y_label = 'Overlapped Volume Ratio'

markerB_x_axis = 'Shortest Distance to MV'
markerB_y_axis = 'Overlap Volume Ratio to MV'

# Create 1x2 subplots
fig, axes = plt.subplots(2, 2, figsize=(10, 8))  # rows, cols

# KDE plots in subplots
sns.kdeplot(x=df_markerB_All[markerB_x_axis], y=df_markerB_All[markerB_y_axis], ax=axes[0, 0], fill=True, 
            cmap='viridis', thresh=0.5)
axes[0, 0].set_title("KDE: markerB_All")
#axes[0, 0].set_xlim(-0.5, 1)
#axes[0, 0].set_ylim(-0.2, 1.2)
axes[0, 0].set_xlabel(markerB_x_label)
axes[0, 0].set_ylabel(markerB_y_label)


sns.kdeplot(x=df_markerB_MV[markerB_x_axis], y=df_markerB_MV[markerB_y_axis], ax=axes[1, 0], fill=True, 
            cmap='viridis', thresh=0.5)

axes[1, 0].set_title("KDE: markerB_MV")
#axes[1, 0].set_xlim(-0.5, 1)
#axes[1, 0].set_ylim(-0.2, 1.2)
axes[1, 0].set_xlabel(markerB_x_label)
axes[1, 0].set_ylabel(markerB_y_label)


markerA_x_label = 'Shortest Distance to MV'
markerA_y_label = 'Overlapped Volume Ratio'

markerA_x_axis = 'Shortest Distance to MV'
markerA_y_axis = 'Overlap Volume Ratio to MV'

# KDE plots in subplots
sns.kdeplot(x=df_markerA_All[markerA_x_axis], y=df_markerA_All[markerA_y_axis], ax=axes[0, 1], fill=True, 
            cmap='viridis', thresh=0.5)
axes[0, 1].set_title("KDE: markerA_All")
#axes[0, 1].set_xlim(-0.5, 1)
#axes[0, 1].set_ylim(-0.2, 1.2)
axes[0, 1].set_xlabel(markerA_x_label)
axes[0, 1].set_ylabel(markerA_y_label)

sns.kdeplot(x=df_markerA_MV[markerA_x_axis], y=df_markerA_MV[markerA_y_axis], ax=axes[1, 1], fill=True, 
            cmap='viridis', thresh=0.5)

axes[1, 1].set_title("KDE: markerA_MV")
#axes[1, 1].set_xlim(-0.5, 1)
#axes[1, 1].set_ylim(-0.2, 1.2)
axes[1, 1].set_xlabel(markerA_x_label)
axes[1, 1].set_ylabel(markerA_y_label)

# Optional: improve spacing
plt.tight_layout()

### 15.6 1D KDE plot

In [ ]:
# 
x_label = 'Shortest Distance to MV'
y_label = 'Overlapped Volume Ratio'

x_axis = 'Shortest Distance to MV'
y_axis = 'Overlap Volume Ratio to MV'
z_axis = 'Distance to cell (mean)'

# Create 1x2 subplots
fig, axes = plt.subplots(3, 1, figsize=(10, 8))  # rows, cols

# KDE plots in subplots
sns.kdeplot(df_markerB_All[x_axis], ax=axes[0], label='markerB_All')
sns.kdeplot(df_markerA_All[x_axis], ax=axes[0], label='markerA_All')
axes[0].set_title("KDE: Shortest Distance to MV")
axes[0].set_xlim(-0.4, 2)
axes[0].set_ylim(-0.1, 3.5)
#axes[0, 0].set_xlabel(x_label)
#axes[0, 0].set_ylabel(y_label)
axes[0].legend()
# Overlap Volume Ratio to MV
sns.kdeplot(df_markerB_All[y_axis], ax=axes[1], label='markerB_All')
sns.kdeplot(df_markerA_All[y_axis], ax=axes[1], label='markerA_All')

axes[1].set_title("KDE: Overlap Volume Ratio to MV")
axes[1].set_xlim(-0.3, 1.3)
axes[1].set_ylim(-0.1, 4)
#axes[1].set_xlabel(x_label)
#axes[1].set_ylabel(y_label)
axes[1].legend()

# Distransformation to cell
sns.kdeplot(df_markerB_All[z_axis], ax=axes[2], label='markerB_All')
sns.kdeplot(df_markerA_All[z_axis], ax=axes[2], label='markerA_All')

axes[2].set_title("KDE: Distance to cell (mean)")
axes[2].set_xlim(-0.2, 2)
axes[2].set_ylim(-0.1, 10)
#axes[2].set_xlabel(x_label)
#axes[2].set_ylabel(y_label)
axes[2].legend()


# Optional: improve spacing
plt.tight_layout()

## 16. Cluster distance from cell surface
* Use DT_max of surfaces
* Can tell if clusters distribute on MV tips or base

##### Select clusters with different DT_max cutoff and get the mean DT_max

In [ ]:
# Set DT_max cut-off as x-axis:  
dt_max_thr = np.linspace(0, 1.5, num=16)
# Create an empty DataFrame to store the results
df_cluster_DT = pd.DataFrame(columns=['DT_max cutoff', 
                                      'DT_max markerA mean', 'DT_max markerA SEM', 
                                      'DT_max markerB mean', 'DT_max markerB SEM', 
                                      'DT_max MV mean', 'DT_max MV SEM', 
                                      'Volume markerA mean', 'Volume markerA SEM',
                                      'Volume markerB mean', 'Volume markerB SEM',
                                      'Volume MV mean', 'Volume MV SEM',
                                      'Intensity mean markerA mean', 'Intensity mean markerA SEM',
                                      'Intensity mean markerB mean', 'Intensity mean markerB SEM',
                                      'Percent of surface (N) markerA',
                                      'Percent of surface (N) markerB',
                                      'Percent of surface (N) MV',
                                      'Percent of surface (V) markerA', 
                                      'Percent of surface (V) markerB', 
                                      'Percent of surface (V) MV', 
                                      'Percent of surface (Int sum) markerA', 
                                      'Percent of surface (Int sum) markerB'
                                     ])

# Define a function to calculate sem with data of any size
def safe_sem(x):
    return stats.sem(x) if len(x) > 1 else np.nan

# Calculate data points

for p in dt_max_thr:
    df_markerB_DT= df_markerB_All[(df_markerB_All['Distance to cell (max)'] >= p) & (df_markerB_All['Distance to cell (max)'] <= 1.5)]
    df_markerA_DT = df_markerA_All[(df_markerA_All['Distance to cell (max)'] >= p) & (df_markerA_All['Distance to cell (max)'] <= 1.5)]
    df_MV_DT = df_MV_select[(df_MV_select['Distance to cell (max)'] >= p) & (df_MV_select['Distance to cell (max)'] <= 1.5)]
    df_cluster_DT.loc[len(df_cluster_DT)] = [p, 
                                             df_markerA_DT['Distance to cell (max)'].mean(), safe_sem(df_markerA_DT['Distance to cell (max)']),
                                             df_markerB_DT['Distance to cell (max)'].mean(), safe_sem(df_markerB_DT['Distance to cell (max)']),
                                             df_MV_DT['Distance to cell (max)'].mean(), safe_sem(df_MV_DT['Distance to cell (max)']),
                                             df_markerA_DT['Volume'].mean(), safe_sem(df_markerA_DT['Volume']),
                                             df_markerB_DT['Volume'].mean(), safe_sem(df_markerB_DT['Volume']),
                                             df_MV_DT['Volume'].mean(), safe_sem(df_MV_DT['Volume']),
                                             df_markerA_DT['Intensity Mean markerA'].mean(), safe_sem(df_markerA_DT['Intensity Mean markerA']),
                                             df_markerB_DT['Intensity Mean markerB'].mean(), safe_sem(df_markerB_DT['Intensity Mean markerB']),
                                             df_markerA_DT.shape[0]/df_markerA_All.shape[0],
                                             df_markerB_DT.shape[0]/df_markerB_All.shape[0],
                                             df_MV_DT.shape[0]/df_MV_select.shape[0],
                                             df_markerA_DT['Volume'].sum()/df_markerA_All['Volume'].sum(), 
                                             df_markerB_DT['Volume'].sum()/df_markerB_All['Volume'].sum(), 
                                             df_MV_DT['Volume'].sum()/df_MV_select['Volume'].sum(),
                                             df_markerA_DT['Intensity Sum markerA'].sum()/df_markerA_All['Intensity Sum markerA'].sum(),
                                             df_markerB_DT['Intensity Sum markerB'].sum()/df_markerB_All['Intensity Sum markerB'].sum()
                                            ]

In [ ]:
# Plot DT_max mean by DT cutoff
plt.errorbar(df_cluster_DT['DT_max cutoff'], df_cluster_DT['DT_max markerA mean'], yerr=df_cluster_DT['DT_max markerA SEM'], 
             fmt='o-', markersize=4, label='markerA')
plt.errorbar(df_cluster_DT['DT_max cutoff'], df_cluster_DT['DT_max markerB mean'], yerr=df_cluster_DT['DT_max markerB SEM'], 
             fmt='o-', markersize=4, label='markerB')
plt.errorbar(df_cluster_DT['DT_max cutoff'], df_cluster_DT['DT_max MV mean'], yerr=df_cluster_DT['DT_max MV SEM'], 
             fmt='o-', markersize=4, label='MV')
plt.legend()
plt.title('DT_max Mean')
plt.xlabel('DT_max cutoff')
plt.ylabel('DT_max mean')

In [ ]:
# Plot cluster volume mean by DT_max cutoff
plt.errorbar(df_cluster_DT['DT_max cutoff'], df_cluster_DT['Volume markerA mean'], yerr=df_cluster_DT['Volume markerA SEM'], 
             fmt='o-', markersize=4, label='markerA')
plt.errorbar(df_cluster_DT['DT_max cutoff'], df_cluster_DT['Volume markerB mean'], yerr=df_cluster_DT['Volume markerB SEM'], 
             fmt='o-', markersize=4, label='markerB')
plt.errorbar(df_cluster_DT['DT_max cutoff'], df_cluster_DT['Volume MV mean'], yerr=df_cluster_DT['Volume MV SEM'], 
             fmt='o-', markersize=4, label='MV')
plt.legend()
plt.title('Volume Mean')
plt.xlabel('DT_max cutoff')
plt.ylabel('Volume mean')

In [ ]:
# Plot cluster intensity mean by DT_max cutoff
plt.errorbar(df_cluster_DT['DT_max cutoff'], df_cluster_DT['Intensity mean markerA mean'], yerr=df_cluster_DT['Intensity mean markerA SEM'], 
             fmt='o-', markersize=4, label='markerA')
plt.errorbar(df_cluster_DT['DT_max cutoff'], df_cluster_DT['Intensity mean markerB mean'], yerr=df_cluster_DT['Intensity mean markerB SEM'], 
             fmt='o-', markersize=4, label='markerB')
plt.legend()
plt.title('Intensity mean Mean')
plt.xlabel('DT_max cutoff')
plt.ylabel('Intensity mean mean')

In [ ]:
# Plot percent by DT_max cutoff
plt.plot(df_cluster_DT['DT_max cutoff'], df_cluster_DT['Percent of surface (N) markerA'],
             markersize=4, label='markerA')
plt.plot(df_cluster_DT['DT_max cutoff'], df_cluster_DT['Percent of surface (N) markerB'],
             markersize=4, label='markerB')
plt.plot(df_cluster_DT['DT_max cutoff'], df_cluster_DT['Percent of surface (N) MV'],
             markersize=4, label='MV')
plt.legend()
plt.title('Percent of number of surfaces')
plt.xlabel('DT_max cutoff')
plt.ylabel('Percent')

In [ ]:
# Plot percent of total surface volume by DT_max cutoff
plt.plot(df_cluster_DT['DT_max cutoff'], df_cluster_DT['Percent of surface (V) markerA'],
             markersize=4, label='markerA')
plt.plot(df_cluster_DT['DT_max cutoff'], df_cluster_DT['Percent of surface (V) markerB'],
             markersize=4, label='markerB')
plt.plot(df_cluster_DT['DT_max cutoff'], df_cluster_DT['Percent of surface (V) MV'],
             markersize=4, label='MV')
plt.legend()
plt.title('Percent of total Volume of surfaces')
plt.xlabel('DT_max cutoff')
plt.ylabel('Percent Volume')

In [ ]:
# Plot percent of total intensity sum by DT_max cutoff
plt.plot(df_cluster_DT['DT_max cutoff'], df_cluster_DT['Percent of surface (Int sum) markerA'],
             markersize=4, label='markerA')
plt.plot(df_cluster_DT['DT_max cutoff'], df_cluster_DT['Percent of surface (Int sum) markerB'],
             markersize=4, label='markerB')
plt.legend()
plt.title('Percent of total intensity sum of surfaces')
plt.xlabel('DT_max cutoff')
plt.ylabel('Percent Intensity sum')

## 17. Colocalization plots for markerA Vs markerB

### 17.1 2D KDE plot for markerA Vs markerB

In [ ]:
# 
markerB_x_label = 'Shortest Distance to markerA_MV'
markerB_y_label = 'Overlapped Volume Ratio'

markerB_x_axis = 'Shortest Distance to markerA_MV'
markerB_y_axis = 'Overlap Volume Ratio to markerA_MV'

# Create 1x2 subplots
fig, axes = plt.subplots(2, 2, figsize=(10, 8))  # rows, cols

# KDE plots in subplots
sns.kdeplot(x=df_markerB_All[markerB_x_axis], y=df_markerB_All[markerB_y_axis], ax=axes[0, 0], fill=True, 
            cmap='viridis', thresh=0.5)
axes[0, 0].set_title("KDE: markerB_All")
#axes[0, 0].set_xlim(-0.5, 1)
#axes[0, 0].set_ylim(-0.2, 1.2)
axes[0, 0].set_xlabel(markerB_x_label)
axes[0, 0].set_ylabel(markerB_y_label)


sns.kdeplot(x=df_markerB_MV[markerB_x_axis], y=df_markerB_MV[markerB_y_axis], ax=axes[1, 0], fill=True, 
            cmap='viridis', thresh=0.5)

axes[1, 0].set_title("KDE: markerB_MV")
#axes[1, 0].set_xlim(-0.5, 1)
#axes[1, 0].set_ylim(-0.2, 1.3)
axes[1, 0].set_xlabel(markerB_x_label)
axes[1, 0].set_ylabel(markerB_y_label)


markerA_x_label = 'Shortest Distance to markerB_MV'
markerA_y_label = 'Overlapped Volume Ratio'

markerA_x_axis = 'Shortest Distance to markerB_MV'
markerA_y_axis = 'Overlap Volume Ratio to markerB_MV'

# KDE plots in subplots
sns.kdeplot(x=df_markerA_All[markerA_x_axis], y=df_markerA_All[markerA_y_axis], ax=axes[0, 1], fill=True, 
            cmap='viridis', thresh=0.5)
axes[0, 1].set_title("KDE: markerA_All")
#axes[0, 1].set_xlim(-0.5, 1)
#axes[0, 1].set_ylim(-0.2, 1.2)
axes[0, 1].set_xlabel(markerA_x_label)
axes[0, 1].set_ylabel(markerA_y_label)

sns.kdeplot(x=df_markerA_MV[markerA_x_axis], y=df_markerA_MV[markerA_y_axis], ax=axes[1, 1], fill=True, 
            cmap='viridis', thresh=0.5)

axes[1, 1].set_title("KDE: markerA_MV")
#axes[1, 1].set_xlim(-0.5, 1)
#axes[1, 1].set_ylim(-0.2, 1.3)
axes[1, 1].set_xlabel(markerA_x_label)
axes[1, 1].set_ylabel(markerA_y_label)

# Optional: improve spacing
plt.tight_layout()

## 18. Colocalized clusters

In [ ]:
# filtering parameters
# Set the threshold
pVol_threshold = 0
minDist_threshold = 0

### 18.1 markerA

In [ ]:
# Filter df_markerA_selec for pVol > pVol_threshold & minDist < minDist_threshold
df_markerA_coloc_filter = df_markerA_selec[(df_markerA_selec['Overlap Volume Ratio to markerB_MV'] > pVol_threshold) & (df_markerA_selec['Shortest Distance to markerB_MV'] <= minDist_threshold)]
df_markerA_coloc_filter.shape[0]/df_markerA_selec.shape[0]

In [ ]:
df_markerA_MV_coloc = df_markerA_selec[df_markerA_selec['Overlap Volume Ratio to MV'] > 0]
df_markerA_MV_coloc.shape[0]/df_markerA_selec.shape[0]

In [ ]:
df_markerA_MV_coloc_2 = df_markerA_selec[df_markerA_selec['Shortest Distance to MV'] < 0.1]
df_markerA_MV_coloc_2.shape[0]/df_markerA_selec.shape[0]

### 18.2 markerB

In [ ]:
# Filter df_markerB_selec for pVol > pVol_threshold & minDist < minDist_threshold
df_markerB_coloc_filter = df_markerB_selec[(df_markerB_selec['Overlap Volume Ratio to markerA_MV'] > pVol_threshold) & (df_markerB_selec['Shortest Distance to markerA_MV'] <= minDist_threshold)]
# print(df_markerB_coloc)
df_markerB_coloc_filter.shape[0]/df_markerB_selec.shape[0]

In [ ]:
df_markerB_MV_coloc = df_markerB_selec[df_markerB_selec['Overlap Volume Ratio to MV'] > 0]
df_markerB_MV_coloc.shape[0]/df_markerB_selec.shape[0]

In [ ]:
df_markerB_MV_coloc_2 = df_markerB_selec[df_markerB_selec['Shortest Distance to MV'] < 0.1]
df_markerB_MV_coloc_2.shape[0]/df_markerB_selec.shape[0]

## 19. Store properties of markerA, markerB and MV in dataframes
* df_MV_property
* df_markerA_property
* df_markerB_property

### 19.1 MV Properties

In [ ]:
# DataFrame containing MV properties
df_MV_property = pd.DataFrame({
    'Cell Area': [cell_area],
    'Cell Surface Area': [cell_surf_area],
    'Total Number': [df_MV_select.shape[0]],
    'Density (cell)': [df_MV_select.shape[0]/cell_area],
    'Density (cell surface)': [df_MV_select.shape[0]/cell_surf_area],
    'Volume Mean': [df_MV_select['Volume'].mean()],
    'Mean Ellipsoid Axis Long': [df_MV_select['Ellipsoid Axis Length C'].mean()],
    'Mean Ellipsoid Axis Short': [df_MV_select['Ellipsoid Axis Length A'].mean()]
})

### 19.2 markerA Properties

In [ ]:
# DataFrame for markerA property
df_markerA_property = pd.DataFrame({
    'Cell Area': [cell_area],
    'Cell Surface Area': [cell_surf_area],
    'Total Number': [df_markerA_All.shape[0]],
    'Density (cell)': [df_markerA_All.shape[0]/cell_area],
    'Density (cell surface)': [df_markerA_All.shape[0]/cell_surf_area],
    'Volume Mean': [df_markerA_All['Volume'].mean()],
    'Intensity Mean': [df_markerA_All['Intensity Mean markerA'].mean()],
    'Intensity Sum': [df_markerA_All['Intensity Sum markerA'].sum()],
    'Cluster Intensity Density (cell)': [df_markerA_All['Intensity Sum markerA'].sum()/cell_area],
    'Cluster Intensity Density (cell surface)': [df_markerA_All['Intensity Sum markerA'].sum()/cell_surf_area]
})

### 19.3 markerB Properties

In [ ]:
# DataFrame for markerB property
df_markerB_property = pd.DataFrame({
    'Cell Area': [cell_area],
    'Cell Surface Area': [cell_surf_area],
    'Total Number': [df_markerB_All.shape[0]],
    'Density (cell)': [df_markerB_All.shape[0]/cell_area],
    'Density (cell surface)': [df_markerB_All.shape[0]/cell_surf_area],
    'Volume Mean': [df_markerB_All['Volume'].mean()],
    'Intensity Mean': [df_markerB_All['Intensity Mean markerB'].mean()],
    'Intensity Sum': [df_markerB_All['Intensity Sum markerB'].sum()],
    'Cluster Intensity Density (cell)': [df_markerB_All['Intensity Sum markerB'].sum()/cell_area],
    'Cluster Intensity Density (cell surface)': [df_markerB_All['Intensity Sum markerB'].sum()/cell_surf_area]
})

## 20. Colocalization analysis with different methods

### 20.1 Coloc 0: use data from binary markerA, binary markerB, binary MV
* coloc_A_in_B = (sum of overlapped voxels of A&B)/total voxels of B

#### 20.1.1 MV
* markerA in MV
* markerA only in MV --> not applicable for binary mask for all clusters 
* markerB in MV
* markerB only in MV --> not applicable for binary mask for all clusters
* markerA and markerB in MV --> currently do not have this info. Need to do channel arithmatic in Imaris: channel bianry markerA + channel bianry markerB --> filter voxels with intensity =2 
* markerA or markerB in MV --> currently do not have this info. Need to do channel arithmatic in Imaris: channel bianry markerA + channel bianry markerB --> filter voxcels with intensity >=1

In [ ]:
# markerA in MV
voxel_coloc_markerAinMV = df_MV_select['MV in markerA'].sum()/df_MV_select['MV in MV'].sum()
# markerB in MV
voxel_coloc_markerBinMV = df_MV_select['MV in markerB'].sum()/df_MV_select['MV in MV'].sum()

#### 20.1.2 markerA

In [ ]:
# MV in markerA
voxel_coloc_MVinmarkerA = df_markerA_All['Voxel Overlap with MV'].sum()/df_markerA_All['Voxel Overlap with markerA'].sum()
# markerB in markerA
voxel_coloc_markerBinmarkerA = df_markerA_All['Voxel Overlap with markerB'].sum()/df_markerA_All['Voxel Overlap with markerA'].sum()

#### 20.1.3 markerB

In [ ]:
# MV in markerB
voxel_coloc_MVinmarkerB = df_markerB_All['Voxel Overlap with MV'].sum()/df_markerB_All['Voxel Overlap with markerB'].sum()
# markerA in markerB
voxel_coloc_markerAinmarkerB = df_markerB_All['Voxel Overlap with markerA'].sum()/df_markerB_All['Voxel Overlap with markerB'].sum()

#### 20.1.4 Coloc 0: cluster coloc from binary markerA and binary markerB (old way in Imaris)

In [ ]:
# MV in markerA
clust_coloc_MVinmarkerA = df_markerA_All['Cluster Voxel Overlap with MV'].mean()
print(clust_coloc_MVinmarkerA)

# MV in markerB
clust_coloc_MVinmarkerB = df_markerB_All['Cluster Voxel Overlap with MV'].mean()
print(clust_coloc_MVinmarkerB)

In [ ]:
# markerA in markerB
clust_coloc_markerAinmarkerB = df_markerB_All['Cluster Voxel Overlap with markerA'].mean()
print(clust_coloc_markerAinmarkerB)

# markerA in MV
cluster_coloc_markerAinMV = df_MV_select['Cluster Voxel Overlap with markerA'].mean()
print(cluster_coloc_markerAinMV)

In [ ]:
# markerB in markerA
clust_coloc_markerBinmarkerA = df_markerA_All['Cluster Voxel Overlap with markerB'].mean()
print(clust_coloc_markerBinmarkerA)

# markerB in MV
cluster_coloc_markerBinMV = df_MV_select['Cluster Voxel Overlap with markerB'].mean()
print(cluster_coloc_markerBinMV)

In [ ]:
# Save Data as DataFrame
df_coloc_binary_mask = pd.DataFrame({
    'markerA in MV':[voxel_coloc_markerAinMV],
    'markerB in MV': [voxel_coloc_markerBinMV],
    'MV in markerA': [voxel_coloc_MVinmarkerA],
    'markerB in markerA': [voxel_coloc_markerBinmarkerA],
    'MV in markerB': [voxel_coloc_MVinmarkerB],
    'markerA in markerB': [voxel_coloc_markerAinmarkerB],
    'cluster overlap: markerA in MV': [cluster_coloc_markerAinMV],
    'cluster overlap: markerB in MV': [cluster_coloc_markerBinMV],
    'cluster overlap: MV in markerA': [clust_coloc_MVinmarkerA],
    'cluster overlap: MV in markerB': [clust_coloc_MVinmarkerB],
    'cluster overlap: markerA in markerB': [clust_coloc_markerAinmarkerB],
    'cluster overlap: markerB in markerA': [clust_coloc_markerBinmarkerA]   
})

### 20.2 Coloc 1: use data from df_markerA_selec, and df_markerB_selec

#### 20.2.1 MV

In [ ]:
# markerA+ MV
# Percentage of number of MV that has nearest distance to markerA_selec surfaces <= 0
obj_coloc_markerA_MV = df_MV_select[df_MV_select['Shortest Distance to markerA_s'] <= 0].shape[0]/df_MV_select.shape[0]
# Percentage of total volume of MV that has nearest distance to markerA_selec surfaces <= 0
obj_coloc_markerA_MV_volume = df_MV_select[df_MV_select['Shortest Distance to markerA_s'] <= 0]['Volume'].sum()/df_MV_select['Volume'].sum()
# markerB+ MV
# Percentage of MV that has nearest distance to markerB_selec surfaces <=0
obj_coloc_markerB_MV = df_MV_select[df_MV_select['Shortest Distance to markerB_s'] <= 0].shape[0]/df_MV_select.shape[0]
# Percentage of total volume of MV that has nearest distance to markerB_selec surfaces <=0
obj_coloc_markerB_MV_volume = df_MV_select[df_MV_select['Shortest Distance to markerB_s'] <= 0]['Volume'].sum()/df_MV_select['Volume'].sum()
# receptor+ MV (markerA or markerB)
# Percentage of MV that has nearest distance to markerB_selec surfaces OR markerA_selec surfaces <=0 
obj_coloc_markerA_OR_markerB_MV = df_MV_select[(df_MV_select['Shortest Distance to markerA_s'] <= 0) | (df_MV_select['Shortest Distance to markerB_s'] <= 0)].shape[0]/df_MV_select.shape[0]
# markerA+markerB+ MV 
obj_coloc_markerA_AND_markerB_MV = df_MV_select[(df_MV_select['Shortest Distance to markerA_s'] <= 0) & (df_MV_select['Shortest Distance to markerB_s'] <= 0)].shape[0]/df_MV_select.shape[0]
# markerA single+ MV
obj_coloc_markerA_no_markerB_MV = df_MV_select[(df_MV_select['Shortest Distance to markerA_s'] <= 0) & (df_MV_select['Shortest Distance to markerB_s'] > 0)].shape[0]/df_MV_select.shape[0]
# markerB single+ MV
obj_coloc_markerB_no_markerA_MV = df_MV_select[(df_MV_select['Shortest Distance to markerB_s'] <= 0) & (df_MV_select['Shortest Distance to markerA_s'] > 0)].shape[0]/df_MV_select.shape[0]

In [ ]:
df_MV_select[df_MV_select['Shortest Distance to markerA_s'] <= 0]['Volume'].sum()

#### 20.2.2 markerA

In [ ]:
# MV+ markerA clusters
# Percentage of number of markerA_selec that has nearest distance to MV surfaces <= 0
obj_coloc_MV_markerA = df_markerA_selec[df_markerA_selec['Shortest Distance to MV'] <= 0].shape[0]/df_markerA_selec.shape[0]
# markerB_MV+ markerA clusters
obj_coloc_markerB_markerA = df_markerA_selec[df_markerA_selec['Shortest Distance to markerB_MV'] <= 0].shape[0]/df_markerA_selec.shape[0]

# Percentage of volume of markerA_selec that has nearest distance to MV surfaces <= 0
obj_coloc_MV_markerA_volume = df_markerA_selec[df_markerA_selec['Shortest Distance to MV'] <= 0]['Volume'].sum()/df_markerA_selec['Volume'].sum()
# markerB_MV+ markerA clusters
obj_coloc_markerB_markerA_volume = df_markerA_selec[df_markerA_selec['Shortest Distance to markerB_MV'] <= 0]['Volume'].sum()/df_markerA_selec['Volume'].sum()

# Percentage of intensity of markerA_selec that has nearest distance to MV surfaces <= 0
obj_coloc_MV_markerA_int = df_markerA_selec[df_markerA_selec['Shortest Distance to MV'] <= 0]['Intensity Sum markerA'].sum()/df_markerA_selec['Intensity Sum markerA'].sum()
# markerB_MV+ markerA clusters
obj_coloc_markerB_markerA_int = df_markerA_selec[df_markerA_selec['Shortest Distance to markerB_MV'] <= 0]['Intensity Sum markerA'].sum()/df_markerA_selec['Intensity Sum markerA'].sum()

In [ ]:
df_markerA_selec.columns

#### 20.2.3 markerB

In [ ]:
# MV+ markerB clusters
# Percentage of number of markerB_selec that has nearest distance to MV surfaces <= 0
obj_coloc_MV_markerB = df_markerB_selec[df_markerB_selec['Shortest Distance to MV'] <= 0].shape[0]/df_markerB_selec.shape[0]
# markerA_MV+ markerB clusters
obj_coloc_markerA_markerB = df_markerB_selec[df_markerB_selec['Shortest Distance to markerA_MV'] <= 0].shape[0]/df_markerB_selec.shape[0]

# Percentage of volume of markerB_selec that has nearest distance to MV surfaces <= 0
obj_coloc_MV_markerB_volume = df_markerB_selec[df_markerB_selec['Shortest Distance to MV'] <= 0]['Volume'].sum()/df_markerB_selec['Volume'].sum()
# markerA_MV+ markerB clusters
obj_coloc_markerA_markerB_volume = df_markerB_selec[df_markerB_selec['Shortest Distance to markerA_MV'] <= 0]['Volume'].sum()/df_markerB_selec['Volume'].sum()

# Percentage of total intensity of markerB_selec that has nearest distance to MV surfaces <= 0
obj_coloc_MV_markerB_int = df_markerB_selec[df_markerB_selec['Shortest Distance to MV'] <= 0]['Intensity Sum markerB'].sum()/df_markerB_selec['Intensity Sum markerB'].sum()
# markerA_MV+ markerB clusters
obj_coloc_markerA_markerB_int = df_markerB_selec[df_markerB_selec['Shortest Distance to markerA_MV'] <= 0]['Intensity Sum markerB'].sum()/df_markerB_selec['Intensity Sum markerB'].sum()

In [ ]:
# Save Data in DataFrame
df_coloc_surface_select = pd.DataFrame({
    'markerA+ MV': [obj_coloc_markerA_MV],
    'markerA+ MV (volume)': [obj_coloc_markerA_MV_volume],
    'markerB+ MV': [obj_coloc_markerB_MV],
    'markerB+ MV (volume)': [obj_coloc_markerB_MV_volume],
    'MV with at least on receptor': [obj_coloc_markerA_OR_markerB_MV],
    'markerA+ markerB+ MV': [obj_coloc_markerA_AND_markerB_MV],
    'markerA only MV': [obj_coloc_markerA_no_markerB_MV],
    'markerB only MV':[obj_coloc_markerB_no_markerA_MV],
    'MV+ markerA': [obj_coloc_MV_markerA],
    'MV+ markerA (volume)': [obj_coloc_MV_markerA_volume],
    'MV+ markerA (intensity)': [obj_coloc_MV_markerA_int],
    'markerB+ markerA': [obj_coloc_markerB_markerA],
    'markerB+ markerA (volume)': [obj_coloc_markerB_markerA_volume],
    'markerB+ markerA (intensity)': [obj_coloc_markerB_markerA_int],
    'MV+ markerB': [obj_coloc_MV_markerB],
    'MV+ markerB (volume)': [obj_coloc_MV_markerB_volume],
    'MV+ markerB (intensity)': [obj_coloc_MV_markerB_int],
    'markerA+ markerB': [obj_coloc_markerA_markerB],
    'markerA+ markerB (volume)': [obj_coloc_markerA_markerB_volume],
    'markerA+ markerB (intensity)': [obj_coloc_markerA_markerB_int]
})

### 20.3 Coloc 2: use data from df_markerA_All, and df_markerB_All

#### 20.3.1 MV

In [ ]:
# markerA+ MV
# Percentage of number of MV that has nearest distance to markerA_all surfaces <= 0
obj_coloc_markerA_MV_all = df_MV_select[df_MV_select['Shortest Distance to markerA_all'] <= 0].shape[0]/df_MV_select.shape[0]
# Percentage of total volume of MV that has nearest distance to markerA_all surfaces <= 0
obj_coloc_markerA_MV_volume_all = df_MV_select[df_MV_select['Shortest Distance to markerA_all'] <= 0]['Volume'].sum()/df_MV_select['Volume'].sum()
# markerB+ MV
# Percentage of MV that has nearest distance to markerB_all surfaces <=0
obj_coloc_markerB_MV_all = df_MV_select[df_MV_select['Shortest Distance to markerB_all'] <= 0].shape[0]/df_MV_select.shape[0]
# Percentage of total volume of MV that has nearest distance to markerB_all surfaces <=0
obj_coloc_markerB_MV_volume_all = df_MV_select[df_MV_select['Shortest Distance to markerB_all'] <= 0]['Volume'].sum()/df_MV_select['Volume'].sum()
# receptor+ MV (markerA or markerB)
# Percentage of MV that has nearest distance to markerB_all surfaces OR markerA_all surfaces <=0 
obj_coloc_markerA_OR_markerB_MV_all = df_MV_select[(df_MV_select['Shortest Distance to markerA_all'] <= 0) | (df_MV_select['Shortest Distance to markerB_all'] <= 0)].shape[0]/df_MV_select.shape[0]
# markerA+markerB+ MV 
obj_coloc_markerA_AND_markerB_MV_all = df_MV_select[(df_MV_select['Shortest Distance to markerA_all'] <= 0) & (df_MV_select['Shortest Distance to markerB_all'] <= 0)].shape[0]/df_MV_select.shape[0]
# markerA single+ MV
obj_coloc_markerA_no_markerB_MV_all = df_MV_select[(df_MV_select['Shortest Distance to markerA_all'] <= 0) & (df_MV_select['Shortest Distance to markerB_all'] > 0)].shape[0]/df_MV_select.shape[0]
# markerB single+ MV
obj_coloc_markerB_no_markerA_MV_all = df_MV_select[(df_MV_select['Shortest Distance to markerB_all'] <= 0) & (df_MV_select['Shortest Distance to markerA_all'] > 0)].shape[0]/df_MV_select.shape[0]

#### 20.3.2 markerA

In [ ]:
# MV+ markerA clusters
# Percentage of number of markerA_All that has nearest distance to MV surfaces <= 0
obj_coloc_MV_markerA_all = df_markerA_All[df_markerA_All['Shortest Distance to MV'] <= 0].shape[0]/df_markerA_All.shape[0]
# markerB_MV+ markerA clusters
obj_coloc_markerB_markerA_all = df_markerA_All[df_markerA_All['Shortest Distance to markerB_MV'] <= 0].shape[0]/df_markerA_All.shape[0]

# Percentage of volume of markerA_All that has nearest distance to MV surfaces <= 0
obj_coloc_MV_markerA_volume_all = df_markerA_All[df_markerA_All['Shortest Distance to MV'] <= 0]['Volume'].sum()/df_markerA_All['Volume'].sum()
# markerB_MV+ markerA clusters
obj_coloc_markerB_markerA_volume_all = df_markerA_All[df_markerA_All['Shortest Distance to markerB_MV'] <= 0]['Volume'].sum()/df_markerA_All['Volume'].sum()

# Percentage of intensity of markerA_All that has nearest distance to MV surfaces <= 0
obj_coloc_MV_markerA_int_all = df_markerA_All[df_markerA_All['Shortest Distance to MV'] <= 0]['Intensity Sum markerA'].sum()/df_markerA_All['Intensity Sum markerA'].sum()
# markerB_MV+ markerA clusters
obj_coloc_markerB_markerA_int_all = df_markerA_All[df_markerA_All['Shortest Distance to markerB_MV'] <= 0]['Intensity Sum markerA'].sum()/df_markerA_All['Intensity Sum markerA'].sum()

#### 20.3.3 markerB

In [ ]:
# MV+ markerB clusters
# Percentage of number of markerB_All that has nearest distance to MV surfaces <= 0
obj_coloc_MV_markerB_all = df_markerB_All[df_markerB_All['Shortest Distance to MV'] <= 0].shape[0]/df_markerB_All.shape[0]
# markerA_MV+ markerB clusters
obj_coloc_markerA_markerB_all = df_markerB_All[df_markerB_All['Shortest Distance to markerA_MV'] <= 0].shape[0]/df_markerB_All.shape[0]

# Percentage of volume of markerB_All that has nearest distance to MV surfaces <= 0
obj_coloc_MV_markerB_volume_all = df_markerB_All[df_markerB_All['Shortest Distance to MV'] <= 0]['Volume'].sum()/df_markerB_All['Volume'].sum()
# markerA_MV+ markerB clusters
obj_coloc_markerA_markerB_volume_all = df_markerB_All[df_markerB_All['Shortest Distance to markerA_MV'] <= 0]['Volume'].sum()/df_markerB_All['Volume'].sum()

# Percentage of total intensity of markerB_All that has nearest distance to MV surfaces <= 0
obj_coloc_MV_markerB_int_all = df_markerB_All[df_markerB_All['Shortest Distance to MV'] <= 0]['Intensity Sum markerB'].sum()/df_markerB_All['Intensity Sum markerB'].sum()
# markerA_MV+ markerB clusters
obj_coloc_markerA_markerB_int_all = df_markerB_All[df_markerB_All['Shortest Distance to markerA_MV'] <= 0]['Intensity Sum markerB'].sum()/df_markerB_All['Intensity Sum markerB'].sum()

In [ ]:
# Save Data in DataFrame
df_coloc_surface_all = pd.DataFrame({
    'markerA+ MV': [obj_coloc_markerA_MV_all],
    'markerA+ MV (volume)': [obj_coloc_markerA_MV_volume_all],
    'markerB+ MV': [obj_coloc_markerB_MV_all],
    'markerB+ MV (volume)': [obj_coloc_markerB_MV_volume_all],
    'MV with at least on receptor': [obj_coloc_markerA_OR_markerB_MV_all],
    'markerA+ markerB+ MV': [obj_coloc_markerA_AND_markerB_MV_all],
    'markerA only MV': [obj_coloc_markerA_no_markerB_MV_all],
    'markerB only MV':[obj_coloc_markerB_no_markerA_MV_all],
    'MV+ markerA': [obj_coloc_MV_markerA_all],
    'MV+ markerA (volume)': [obj_coloc_MV_markerA_volume_all],
    'MV+ markerA (intensity)': [obj_coloc_MV_markerA_int_all],
    'markerB+ markerA': [obj_coloc_markerB_markerA_all],
    'markerB+ markerA (volume)': [obj_coloc_markerB_markerA_volume_all],
    'markerB+ markerA (intensity)': [obj_coloc_markerB_markerA_int_all],
    'MV+ markerB': [obj_coloc_MV_markerB_all],
    'MV+ markerB (volume)': [obj_coloc_MV_markerB_volume_all],
    'MV+ markerB (intensity)': [obj_coloc_MV_markerB_int_all],
    'markerA+ markerB': [obj_coloc_markerA_markerB_all],
    'markerA+ markerB (volume)': [obj_coloc_markerA_markerB_volume_all],
    'markerA+ markerB (intensity)': [obj_coloc_markerA_markerB_int_all]
})

## 21. Receptor clusters above the cell membrane: 
* use # of surface in df_markerA_selec (p_markerA_select and p_markerB_select)
* percent of surfaces in df_receptor_All that is above the cell surface with a user defined threshold (p_markerA_abv, p_markerB_abv)

In [ ]:
# Define the DT_max for surfaces to be selected
abv_distance = 0.3

### 21.1 markerA

In [ ]:
# % markerA clusters above the cell surface (DT_max >0.2)
p_markerA_select = df_markerA_selec.shape[0]/df_markerA_All.shape[0]
p_markerA_abv = df_markerA_All[df_markerA_All['Distance to cell (max)'] >= abv_distance].shape[0]/df_markerA_All.shape[0]
# % markerA cluster voume above the cell surface (DT_max >0.2)
p_markerA_select_vol = df_markerA_selec['Volume'].sum()/df_markerA_All['Volume'].sum()
p_markerA_abv_vol = df_markerA_All[df_markerA_All['Distance to cell (max)'] >= abv_distance]['Volume'].sum()/df_markerA_All['Volume'].sum()
# # % markerA cluster intensity above the cell surface (DT_max >0.2)
p_markerA_select_int = df_markerA_selec['Intensity Sum markerA'].sum()/df_markerA_All['Intensity Sum markerA'].sum()
p_markerA_abv_int = df_markerA_All[df_markerA_All['Distance to cell (max)'] >= abv_distance]['Intensity Sum markerA'].sum()/df_markerA_All['Intensity Sum markerA'].sum()

In [ ]:
df_markerA_selec.columns

### 21.2 markerB

In [ ]:
# % markerB clusters above the cell surface (DT_max >=0.2)
p_markerB_select = df_markerB_selec.shape[0]/df_markerB_All.shape[0]
p_markerB_abv = df_markerB_All[df_markerB_All['Distance to cell (max)'] >= abv_distance].shape[0]/df_markerB_All.shape[0]
# % markerB clusters volume above the cell surface (DT_max >=0.2)
p_markerB_select_vol = df_markerB_selec['Volume'].sum()/df_markerB_All['Volume'].sum()
p_markerB_abv_vol = df_markerB_All[df_markerB_All['Distance to cell (max)'] >= abv_distance]['Volume'].sum()/df_markerB_All['Volume'].sum()
# % markerB clusters intensity above the cell surface (DT_max >=0.2)
p_markerB_select_int = df_markerB_selec['Intensity Sum markerB'].sum()/df_markerB_All['Intensity Sum markerB'].sum()
p_markerB_abv_int = df_markerB_All[df_markerB_All['Distance to cell (max)'] >= abv_distance]['Intensity Sum markerB'].sum()/df_markerB_All['Intensity Sum markerB'].sum()

In [ ]:
# Save data as DataFrame
df_cluster_abv_surface = pd.DataFrame({
    'Number% of selected markerA cluster': [p_markerA_select],
    'Volume% of selected markerA cluster': [p_markerA_select_vol],
    'Intensity% of selected markerA cluster': [p_markerA_select_int],
    'Number% of selected markerB cluster': [p_markerB_select],
    'Volume% of selected markerB cluster': [p_markerB_select_vol],
    'Intensity% of selected markerB cluster': [p_markerB_select_int],
    'DT_max for surface above': [abv_distance],
    'Number% of markerA cluster above surface': [p_markerA_abv],
    'Volume% of markerA cluster above surface': [p_markerA_abv_vol],
    'Intensity% of markerA cluster above surface': [p_markerA_abv_int],
    'Number% of markerB cluster above surface': [p_markerB_abv],
    'Volume% of markerB cluster above surface': [p_markerB_abv_vol],
    'Intensity% of markerB cluster above surface': [p_markerB_abv_int]
})

## 22. Save results in excel
* Excel file: MV_cluster_analysis.xlsx

In [ ]:
with pd.ExcelWriter(os.path.join(result_folder_path, 'MV_cluster_analysis.xlsx'), engine='xlsxwriter') as writer:
    df_MV_property.to_excel(writer, sheet_name='df_MV_property', index=False)
    df_markerA_property.to_excel(writer, sheet_name='df_markerA_property', index=False)
    df_markerB_property.to_excel(writer, sheet_name='df_markerB_property', index=False)
    df_coloc_binary_mask.to_excel(writer, sheet_name='df_coloc_binary_mask', index=False)
    df_coloc_surface_select.to_excel(writer, sheet_name='df_coloc_surface_select', index=False)
    df_coloc_surface_all.to_excel(writer, sheet_name='df_coloc_surface_all', index=False)
    df_cluster_abv_surface.to_excel(writer, sheet_name='df_cluster_abv_surface', index=False)
    df_cluster_DT.to_excel(writer, sheet_name='df_cluster_DT', index=False)